# Ensnarlment in physical networks

This is to explore the ensnarlment between cycles in multiscale spaces: cycle - edge - node spaces.

The code is largely based on the python modules:
- kirchhoff-circuits: https://github.com/felixk1990/kirchhoff-circuits
- SnarlPy: https://github.com/felixk1990/SnarlPy/tree/main



## Functions

In [ ]:
import numpy as np
import networkx as nx
from dataclasses import dataclass, field

### Data Classes

In [ ]:
@dataclass
class NetworkxCrystal():

    """
    A base class for spatial, crystal-like graphs.

    Attributes
    ----------

        G (nx.Graph):\n
            An internal networkx graph variable.
        dict_cells (dictionary):\n
            A dictionary of the current cells.
        lattice_constant (float):\n
            Scale for the lattice. spacing
        translation_length (float):\n
            Scale for the translational offset.

    """

    def __init__(self):

        """
        A constructor for crystal objects, setting default values
        for the interal graph objects and geometry
        """

        self.G = nx.Graph()
        self.dict_cells = {}
        self.lattice_constant = 1
        self.translation_length = 1

    # construct one of the following crystal topologies
    def lattice_translation(self, t, T):

        """
        Return a networkx graph, initialzed from given unit cell and offset.

        Args:
            t (ndarray):\n
                A translational offset for the lattice.
            T (nx.Graph):\n
                A networkx graph, unit cell.

        Returns:
            nx.Graph: A simple, periodic Graph.

        """

        D = nx.Graph()
        for n in T.nodes():
            D.add_node(tuple(n+t), pos=T.nodes[n]['pos']+t)

        return D

    def periodic_cell_structure(self, cell, num_periods):

        """
        Set the internal graph variable by periodically repeating the chosen
        unitcell type.

        Args:
            cell (nx.Graph):\n
                A networkx graph, unit cell. (only with nodes)
            num_periods (int):\n
                Repetition number for the unit cells.

        """

        DL = nx.Graph()

        if type(num_periods) is not int:
            periods = [range(num_periods[i]) for i in range(3)] # different in x,y,z
        else:
            periods = [range(num_periods) for i in range(3)] # same in all dimensions
        for i in periods[0]:
            for j in periods[1]:
                for k in periods[2]:
                    v = self.translation_length*np.array([i, j, k])
                    TD = self.lattice_translation(v, cell)
                    DL.add_nodes_from(TD.nodes(data=True))
                    self.dict_cells[(i, j, k)] = list(TD.nodes())

        # add edges
        list_n = np.array(list(DL.nodes()))
        for i, n in enumerate(list_n[:-1]):
            for m in list_n[(i+1):]:
                p = DL.nodes[tuple(n)]['pos']
                q = DL.nodes[tuple(m)]['pos']
                dist = np.linalg.norm(p-q)
                if dist == self.lattice_constant:
                    DL.add_edge(tuple(n), tuple(m))

        dict_nodes = {}
        for idx_n, n in enumerate(DL.nodes()):
            self.G.add_node(idx_n, pos=DL.nodes[n]['pos'])
            dict_nodes.update({n: idx_n})
        for idx_e, e in enumerate(DL.edges()):
            u, v = dict_nodes[e[0]], dict_nodes[e[1]]
            self.G.add_edge(u, v)

        self.dict_cubes = {}
        dict_aux = {}
        for i, k in enumerate(self.dict_cells.keys()):
            dict_aux[i] = [dict_nodes[n] for n in self.dict_cells[k]]
        for i, k in enumerate(dict_aux.keys()):
            self.dict_cubes[k] = nx.Graph()
            n_list = list(dict_aux[k])
            for u in n_list[:-1]:
                for v in n_list[1:]:
                    if self.G.has_edge(u, v):
                        self.dict_cubes[k].add_edge(u, v)


# 3D
@dataclass
class NetworkxSimple(NetworkxCrystal):

    """
    A derived class for spatial, simple cubic graphs.

    """

    def __init__(self,  num_periods):

        """
        A constructor for simple cubic crystal objects, setting default values
        for the interal graph objects and geometry
        """

        super(NetworkxSimple, self).__init__()
        self.lattice_constant = 1.
        self.translation_length = 1.
        self.simple_cubic_lattice(num_periods)

    # construct full cubic grid as skeleton
    def simple_unit_cell(self):

        """
        Return a networkx graph of the simbple cubic unit cell.

        Returns:
            nx.Graph:\n
                A  networkx graph.

        """

        D = nx.Graph()
        for i in [0, 1]:
            for j in [0, 1]:
                for k in [0, 1]:
                    D.add_node(tuple((i, j, k)), pos=np.array([i, j, k]))

        return D

    def simple_cubic_lattice(self, num_periods):

        """
        Set the internal graph as simple cubic lattice.

        Args:
            num_periods (int):\n
                Repetition number for the unit cell.

        """

        D = self.simple_unit_cell()
        self.periodic_cell_structure(D, num_periods)

@dataclass
class NetworkxChain(NetworkxCrystal):

    """
    A derived class for spatial, 1D chain graphs.

    """

    def __init__(self, num_periods):

        """
        A constructor for chain objects, setting default values
        for the interal graph objects and geometry
        """

        super(NetworkxChain, self).__init__()
        self.simple_chain(num_periods)

    def simple_chain(self, num_periods):

        """
        Set the internal graph as a simple 1D chain.

        Args:
            num_periods (int):\n
                Length of the chain.

        """

        # construct single box
        for i in range(num_periods):
            self.G.add_node(i, pos=np.array([i, 0, 0]))
        for i in range(num_periods-1):
            self.G.add_edge(i+1, i)


class NetworkxBcc(NetworkxCrystal):

    """
    A derived class for spatial, simple bcc graphs.

    """

    def __init__(self,  num_periods):

        """
        A constructor for simple bcc crystal objects, setting default values
        for the interal graph objects and geometry
        """
        super(NetworkxBcc, self).__init__()
        self.lattice_constant = np.sqrt(3.)/2.
        self.translation_length = 1.
        self.simple_bcc_lattice(num_periods)

    def bcc_unit_cell(self):

        """
        Return a networkx graph of the simple bcc unit cell.

        Returns:
            nx.Graph:\n
                A  networkx graph.

        """

        D = nx.Graph()
        for i in [0, 1]:
            for j in [0, 1]:
                for k in [0, 1]:
                    D.add_node(tuple((i, j, k)), pos=np.array([i, j, k]))
        D.add_node(tuple((0.5, 0.5, 0.5)), pos=np.array([0.5, 0.5, 0.5]))

        return D

    def simple_bcc_lattice(self, num_periods):

        """
        Set the internal graph as simple bcc lattice.

        Args:
            num_periods (int):\n
                Repetition number for the unit cell.

        """

        # construct single box
        D = self.bcc_unit_cell()
        self.periodic_cell_structure(D, num_periods)


class NetworkxFcc(NetworkxCrystal):

    """
    A derived class for spatial, simple fcc graphs.

    """

    def __init__(self, num_periods):

        """
        A constructor for simple fcc crystal objects, setting default values
        for the interal graph objects and geometry
        """
        super(NetworkxFcc, self).__init__()
        self.lattice_constant = np.sqrt(2.)/2.
        self.translation_length = 1.
        self.simple_fcc_lattice(num_periods)

    def fcc_unit_cell(self):

        """
        Return a networkx graph of the simple fcc unit cell.

        Returns:
            nx.Graph:\n
                A  networkx graph.

        """

        D = nx.Graph()
        for i in [0, 1]:
            for j in [0, 1]:
                for k in [0, 1]:
                    D.add_node(tuple((i, j, k)), pos=np.array([i, j, k]))
        for i in [0., 1.]:
            D.add_node(tuple((0.5, i, 0.5)), pos=np.array([0.5, i, 0.5]))
        for i in [0., 1.]:
            D.add_node(tuple((0.5, 0.5, i)), pos=np.array([0.5, 0.5, i]))
        for i in [0., 1.]:
            D.add_node(tuple((i, 0.5, 0.5)), pos=np.array([i, 0.5, 0.5]))

        return D

    def simple_fcc_lattice(self, num_periods):

        """
        Set the internal graph as simple fcc lattice.

        Args:
            num_periods (int):\n
                Repetition number for the unit cell.

        """

        D = self.fcc_unit_cell()
        self.periodic_cell_structure(D, num_periods)


class NetworkxDiamond(NetworkxCrystal):

    """
    A derived class for spatial, diamond graphs.

    """

    def __init__(self, num_periods):
        """
        A constructor for diamond crystal objects, setting default values
        for the interal graph objects and geometry
        """
        super(NetworkxDiamond, self).__init__()
        self.lattice_constant = np.sqrt(3.)/2.
        self.translation_length = 2.
        self.diamond_lattice(num_periods)

    def diamond_unit_cell(self):

        """
        Return a networkx graph of the dioamond unit cell.

        Returns:
            nx.Graph:\n
                A  networkx graph.

        """

        D = nx.Graph()
        T = [nx.Graph() for i in range(4)]
        T[0].add_node((0, 0, 0), pos=np.array([0, 0, 0]))
        T[0].add_node((1, 1, 0), pos=np.array([1, 1, 0]))
        T[0].add_node((1, 0, 1), pos=np.array([1, 0, 1]))
        T[0].add_node((0, 1, 1), pos=np.array([0, 1, 1]))
        T[0].add_node((0.5, 0.5, 0.5), pos=np.array([0.5, 0.5, 0.5]))
        translation = [
            np.array([1, 1, 0]),
            np.array([1, 0, 1]),
            np.array([0, 1, 1])
            ]
        for i, t in enumerate(translation):
            for n in T[0].nodes():
                T[i+1].add_node(tuple(n+t), pos=T[0].nodes[n]['pos']+t)
        for t in T:
            D.add_nodes_from(t.nodes(data=True))

        return D

    def diamond_lattice(self, num_periods):

        """
        Set the internal graph as diamond lattice.

        Args:
            num_periods (int):\n
                Repetition number for the unit cell.

        """

        D = self.diamond_unit_cell()
        self.periodic_cell_structure(D, num_periods)


class NetworkxLaves(NetworkxCrystal):

    """
    A derived class for spatial, Laves graphs.

    """

    def __init__(self, num_periods):
        """
        A constructor for laves crystal objects, setting default values
        for the interal graph objects and geometry.
        """
        super(NetworkxLaves, self).__init__()
        self.lattice_constant = 2.
        self.laves_lattice(num_periods)

    def laves_lattice(self, num_periods):

        """
        Set the internal graph as laves lattice.

        Args:
            num_periods (int):\n
                Repetition number for the unit cell.

        """

        # construct single box
        G_aux = nx.Graph()
        if type(num_periods) is not int:
            periods = [range(num_periods[i]) for i in range(3)]
        else:
            periods = [range(num_periods) for i in range(3)]

        fundamental_points = [
            [0, 0, 0],
            [1, 1, 0],
            [1, 2, 1],
            [0, 3, 1],
            [2, 2, 2],
            [3, 3, 2],
            [3, 0, 3],
            [2, 1, 3]
        ]

        for fp in fundamental_points:
            for i in periods[0]:
                for j in periods[1]:
                    for k in periods[2]:

                        pos_n = np.add(fp, [4.*i, 4.*j, 4.*k])
                        G_aux.add_node(tuple(pos_n), pos=pos_n)

        list_nodes = list(G_aux.nodes())
        self.G = nx.Graph()
        H = nx.Graph()

        for i, n in enumerate(G_aux.nodes()):
            H.add_node(n, pos=G_aux.nodes[n]['pos'])

        for i, n in enumerate(list_nodes[:-1]):
            for j, m in enumerate(list_nodes[(i+1):]):

                v = np.subtract(n, m)
                dist = np.dot(v, v)
                if dist == self.lattice_constant:
                    H.add_edge(n, m)

        dict_nodes = {}
        for idx_n, n in enumerate(H.nodes()):
            self.G.add_node(idx_n, pos=H.nodes[n]['pos'])
            dict_nodes.update({n: idx_n})

        for idx_e, e in enumerate(H.edges()):
            u, v = dict_nodes[e[0]], dict_nodes[e[1]]
            self.G.add_edge(u, v)


class NetworkxTriagonalStack(NetworkxCrystal):

    """
    A derived class for spatial, stacked, triangulated graphs, contained in
    hexagonal shapes.

    """

    def __init__(self, stacks, tiling_factor):
        """
        A constructor for stacked, triangulated crystal objects, setting
        default values for the interal graph objects and geometry.
        """
        super(NetworkxTriagonalStack, self).__init__()
        self.triangulated_hexagon_stack(stacks, tiling_factor)

    # define crosslinking procedure between the generated single-layers
    def crosslink_stacks(self):

        """
        Crosslink the stacked hexagons to closest layer neighbor.
        """

        for i, n in enumerate(self.G.nodes()):
            self.G.nodes[n]['label'] = i

        if self.stacks > 1:

            labels_n = nx.get_node_attributes(self.G, 'label')
            sorted_label_n_list = sorted(labels_n, key=labels_n.__getitem__)

            for n in sorted_label_n_list:
                if n[2] != self.stacks-1:

                    u = (n[0], n[1], n[2])
                    v = (n[0], n[1], n[2]+1)

                    self.G.add_edge(u, v)

    # auxillary function,  construct triangulated hex grid upper and lower
    # wings
    def construct_spine_stack(self, z, n):

        """
        Generate new nodes and connections for spines of stacked hexagons of
         the internal graph and set spine length internally.

        Args:
            z (float):\n
                The current stack indicator.
            n (int):\n
                Length of the hexagon's outer sites.

        """

        self.spine = 2*(n-1)
        self.G.add_node((0, 0, z), pos=(0., 0., z))
        for m in range(self.spine):

            self.G.add_node((m+1, 0, z), pos=((m+1), 0., z))
            u = (m, 0, z)
            v = (m+1, 0, z)

            self.G.add_edge(u, v)

    def construct_wing_stack(self, z, a, n):

        """
        Generate new nodes and connections from the spines for each stacked
        hexagon.

        Args:
            z (float):\n
                The current stack indicator.
            a (int):\n
                +-1, setting the currently constructed hemisphere.
            n (int):\n
                Length of the hexagon's outer sites.

        """

        for m in range(n-1):
            # m-th floor
            floor_m_nodes = self.spine-(m+1)

            u = (0, a*(m+1), z)
            v = (0, a*m, z)
            w = (1, a*m, z)

            self.G.add_node(u, pos=((m+1)/2., a*(np.sqrt(3.)/2.)*(m+1), z))

            self.G.add_edge(u, v)
            self.G.add_edge(u, w)

            for p in range(floor_m_nodes):
                # add 3-junctions
                u = (p+1, a*(m+1), z)
                v = (p+1, a*m, z)
                w = (p+2, a*m, z)
                x = (p, a*(m+1), z)

                ps = (((p+1)+(m+1)/2.), a*(np.sqrt(3.)/2.)*(m+1), z)
                self.G.add_node(u, pos=ps)

                self.G.add_edge(u, v)
                self.G.add_edge(u, w)
                self.G.add_edge(u, x)

    # construct full triangulated hex grids as skeleton of a stacked structure
    def triangulated_hexagon_stack(self, stacks, num_periods):

        """
        Set the internal graph as stacked, triangulated lattice, contained in
        hexagonal shapes.

        Args:
            stacks (int):\n
                The number of layers.
            num_periods (int):\n
                Length of the hexagon's spine.

        """

        self.stacks = stacks
        for z in range(self.stacks):

            # construct spine for different levels of lobule
            self.construct_spine_stack(z, num_periods)

            # construct lower/upper halfspace
            self.construct_wing_stack(z, -1,  num_periods)
            self.construct_wing_stack(z,  1,  num_periods)

        self.crosslink_stacks()


# 2D
class NetworkxSquare(NetworkxCrystal):

    """
    A derived class for spatial, simpley tiled graphs.

    """

    def __init__(self, tiling_factor):

        """
        A constructor for simple tiled crystal objects, setting default values
        for the interal graph objects and geometry
        """
        super(NetworkxSquare, self).__init__()
        self.square_grid(tiling_factor)

    def square_grid(self, num_periods):

        """
        Set the internal graph as square grid.

        """
        # more than 1 period
        if type(num_periods) is not int:
            a = [range(0, num_periods[0]+1), range(0, num_periods[1]+1)]
        else:
            a = [range(0, num_periods+1), range(0, num_periods+1)]

        for x in a[0]:
            for y in a[1]:
                self.G.add_node((x, y), pos=(x, y, 0))

        list_n = list(self.G.nodes())
        dict_d = {}
        threshold = 1.
        for idx_n, n in enumerate(list_n[:-1]):
            for m in list_n[idx_n+1:]:
                p = np.array(self.G.nodes[n]['pos'])
                q = np.array(self.G.nodes[m]['pos'])
                dict_d[(n, m)] = np.linalg.norm(p-q)
                if dict_d[(n,m)] <= threshold:
                    self.G.add_edge(n, m)
        # for nm in dict_d:
        #     if dict_d[nm] <= threshold:
        #         self.G.add_edge(*nm)


class NetworkxTriagonalPlanar(NetworkxCrystal):

    """
    A derived class for spatial, planar triangulated graphs.

    """

    def __init__(self,  tiling_factor):
        """
        A constructor for a planar triangulated crystal objects, setting
        default values for the interal graph objects and geometry
        """
        super(NetworkxTriagonalPlanar, self).__init__()
        self.triangulated_hexagon_lattice(tiling_factor)
    # I) construct and define one-layer hex
    # auxillary function,  construct triangulated hex grid upper and lower
    # wings

    def construct_wing(self, a, n):

        """
        Generate new nodes and connections from the spines of the hexagon.

        Args:
            a (int):\n
                +-1, setting the currently constructed hemisphere.
            n (int):\n
                Length of the hexagon's outer sites.

        """
        for m in range(n-1):
            # m-th floor
            floor_m_nodes = self.spine - (m+1)

            u = (0, a*(m+1))
            v = (0, a*m)
            w = (1, a*m)
            ps = np.array([(m+1)/2., a*(np.sqrt(3.)/2.)*(m+1)])

            self.G.add_node(u, pos=ps)
            self.G.add_edge(u, v)
            self.G.add_edge(u, w)

            for p in range(floor_m_nodes):
                # add 3-junctions
                u = (p+1, a*(m+1))
                v = (p+1, a*m)
                w = (p+2, a*m)
                x = (p, a*(m+1))
                ps = np.array([((p+1)+(m+1)/2.), a*(np.sqrt(3.)/2.)*(m+1)])

                self.G.add_node(u, pos=ps)

                self.G.add_edge(u, v)
                self.G.add_edge(u, w)
                self.G.add_edge(u, x)

    # construct full triangulated hex grid as skeleton
    def triangulated_hexagon_lattice(self, n):

        """
        Generate new nodes and connections for the spine of the hexagon.

        Args:
            n (int):\n
                Length of the hexagon's outer sites.

        """

        # construct spine
        self.spine = 2*(n-1)
        self.G.add_node((0, 0), pos=np.array([0., 0.]))

        for m in range(self.spine):

            self.G.add_node((m+1, 0), pos=np.array([(m+1), 0.]))
            u, v = (m, 0), (m+1, 0)
            self.G.add_edge(u, v)

        # construct lower/upper halfspace
        self.construct_wing(-1, n)
        self.construct_wing(1, n)


class NetworkxHexagonal(NetworkxCrystal):

    """
    A derived class for spatial, planar hexagonal graphs.

    """

    def __init__(self, tiling_factor, periodic=False):
        """
        A constructor for a planar, hexagonal crystal objects, setting default
        values for the interal graph objects and geometry
        """
        super(NetworkxHexagonal, self).__init__()
        self.hexagonal_grid(tiling_factor, periodic)

    def hexagonal_grid(self, tiling_factor, periodic_bool):

        """
        Set the internal graph as hexagonal grid, using the networkx graph
        generator.
        """

        # generate hexagonal grid
        m = 2*tiling_factor+1
        n = 2*tiling_factor
        opt = {
            'periodic': periodic_bool,
            'with_positions': True
        }
        self.G = nx.hexagonal_lattice_graph(m, n, **opt)

        # set embedding data
        for n in self.G.nodes():
            self.G.nodes[n]['pos'] = np.array(self.G.nodes[n]['pos'])

In [ ]:
###### Network Classes ######
from scipy.spatial import Voronoi

@dataclass
class NetworkxDual(NetworkxCrystal):

    """
    A base class for spatial, dual circuits.

    Attributes
    ----------
        layer (list):\n
            List of the graphs contained in the multilayer network.
        lattice_constant (float):\n
            Scale for the spacing between the networks.
        translation_length (float):\n
            Scale for the translation difference between the multiple networks.

    """
    def __init__(self):

        """
        A constructor for multilayer circuit objects, setting default values
        for the interal graph objects and geometry
        """

        super(NetworkxDual, self).__init__()
        self.layer = [nx.Graph(), nx.Graph()]
        self.lattice_constant = 1
        self.translation_length = 1

    def periodic_cell_structure_offset(self, cell, num_periods, offset):

        """
        Repeat the unit cell with translational offset to create a graph.

        Args:
            cell (nx.Graph): unit cell in networkx graph format.
            num_periods (int): Repetition number for the unit cells.
            offset (ndarray): A translational offset for the lattice.

        Returns:
            nx.Graph: A simple, periodic Graph.

        """

        L = nx.Graph()
        periods = range(num_periods)
        for i in periods:
            for j in periods:
                for k in periods:
                    if (i+j+k) % 2 == 0:
                        v = self.translation_length*np.array([i, j, k])
                        TD = self.lattice_translation(offset+v, cell)
                        L.add_nodes_from(TD.nodes(data=True))

        return L

    def prune_leaves(self, G, H, adj):

        """
        Remove non-affiliated edges in dual graphs.

        Args:
            G(nx.Graph):\n
                A networkx graph.
            H (nx.Graph):\n
                A networkx graph.
            adj (list):\n
                A list of affiliated edge pairs of the two graphs.

        Returns:
            list: A list of networkx graphs.

        """

        K = [G, H]
        for i in range(2):

            adj_x = np.array(adj)[:, i]
            list_e = list(K[i].edges())
            for e in list_e:
                if np.any(np.array(adj_x) == K[i].edges[e]['label']):
                    continue
                else:
                    K[i].remove_edge(*e)

            list_n = list(K[i].nodes())
            for n in list_n:
                if not K[i].degree(n) > 0:
                    K[i].remove_node(n)

        return K[0], K[1]

    def relabel_networkx(self, G1, G2, adj):

        """
        Relabel affiliations and graph attributes.

        Args:
            G(nx.Graph):\n
                A networkx graph.
            H (nx.Graph):\n
                A networkx graph.
            adj (list):\n
                A list of affiliated edge pairs of the two graphs.

        Returns:
            list: A list of networkx graphs.

        """
        K = [G1, G2]
        aff = [{}, {}]
        e_adj = []
        e_adj_idx = []
        P = [nx.Graph(), nx.Graph()]
        dict_P = [[{}, {}, {}], [{}, {}, {}]]

        for i in range(2):

            for idx_n, n in enumerate(K[i].nodes()):
                opt = {
                    'pos': K[i].nodes[n]['pos'],
                    'label': K[i].nodes[n]['label']
                }
                P[i].add_node(idx_n, **opt)
                dict_P[i][0].update({n: idx_n})
                aff[i][idx_n] = []

            for idx_e, e in enumerate(K[i].edges()):
                opt = {
                    # 'slope': [K[i].nodes[e[0]]['pos'],
                    # K[i].nodes[e[1]]['pos']],
                    'label': K[i].edges[e]['label']
                }
                v, u = dict_P[i][0][e[0]], dict_P[i][0][e[1]]

                P[i].add_edge(v, u, **opt)

            for j, e in enumerate(P[i].edges()):
                dict_P[i][1].update({P[i].edges[e]['label']: j})
                dict_P[i][2].update({P[i].edges[e]['label']: e})

        for a in adj:

            e = [dict_P[0][1][a[0]], dict_P[1][1][a[1]]]
            E = [dict_P[0][2][a[0]], dict_P[1][2][a[1]]]
            e_adj.append([e[0], e[1]])
            e_adj_idx.append([E[0], E[1]])

            for i in range(2):
                n0 = E[i][0]
                n1 = E[i][1]
                aff[i][n0].append(a[-(i+1)])
                aff[i][n1].append(a[-(i+1)])

        for i in range(2):

            for key in aff[i].keys():
                aff[i][key] = list(set(aff[i][key]))
                aux = []
                for j in aff[i][key]:
                    aux.append(dict_P[-(i+1)][1][j])
                aff[i][key] = aux

        self.e_adj = e_adj
        self.e_adj_idx = e_adj_idx
        self.n_adj = [aff[0], aff[1]]

        return P

    def set_graph_adjacency(self, G, H):

        """
        Relabel affiliations and graph attributes.

        Args:
            G (nx.Graph):\n
                The inner networkx graph.
            H (nx.Graph):\n
                The outer networkx graph.

        Returns:
            list:  A list of affiliated edge pairs of the two graphs.

        """
        adj = []

        for i, e in enumerate(G.edges()):
            a = np.add(G.nodes[e[0]]['pos'], G.nodes[e[1]]['pos'])
            for j, f in enumerate(H.edges()):
                b = np.add(H.nodes[f[0]]['pos'], H.nodes[f[1]]['pos'])
                c = np.subtract(a, b)
                if np.dot(c, c) == 14.:
                    adj.append([G.edges[e]['label'], H.edges[f]['label']])

        return adj

@dataclass
class NetworkxDualSimple(NetworkxDual):
    """
    A class for spatial, dual cubic circuits.

    Attributes
    ----------

        layer (list):\n
            List of the mutlilayered circuits.
        lattice_constant (float):\n
            Scale for the spacing between the networks.
        translation_length (float):\n
            Scale for the translation difference between the multiple networks.

    """
    def __init__(self, num_periods):

        """
        A constructor for multilayer circuit simple objects, setting default
        values for the interal graph objects and geometry
        """

        super(NetworkxDualSimple, self).__init__()
        self.lattice_constant = 1
        self.translation_length = 1
        self.dualSimple(num_periods)

    def dualSimple(self, num_periods):

        """
        Set internal networks structure, dual cubic.

        Args:
            num_periods (int):\n
                A networkx graph.

        """

        # create primary point cloud with lattice structure
        # creating voronoi cells, with defined ridge structure
        ic = NetworkxSimple(1)
        unit_cell = ic.simple_unit_cell()
        self.periodic_cell_structure(unit_cell, num_periods)
        points = [self.G.nodes[n]['pos'] for i, n in enumerate(self.G.nodes())]
        V = Voronoi(points)

        # construct caged networks from given point clouds, with corresponding
        # adjacency list of edges
        G1, G2 = self.init_graph_nuclei(V)

        adj = self.set_graph_adjacency(V, G1, G2)

        # cut off redundant (non-connected or neigborless) points/edges
        G1, G2 = self.prune_leaves(G1, G2, adj)

        # relabeling network nodes/edges & adjacency-list
        P = self.relabel_networkx(G1, G2, adj)

        self.layer = [P[0], P[1]]

    def init_graph_nuclei(self, V):

        """
        Generate points for a cubic lattice and its dual via Voronois
        tesselation and return dual graph representations.

        Args:
            Voronoi (scipy.spatial.Voronoi):\n
                A Tesselation object.

        Return:
            nx.Graph:\n
                Inner networkx graph.
            nx.Graph:\n
                Outer networkx graph.

        """
        H = nx.Graph()
        G = nx.Graph()
        list_p = np.array(list(V.points))
        list_v = np.array(list(V.vertices))

        counter_n = 0
        for j, v in enumerate(list_v):
            H.add_node(j, pos=v, label=counter_n)
            counter_n += 1

        counter_n = 0
        for j, p in enumerate(list_p):
            G.add_node(j, pos=p, label=counter_n)
            counter_n += 1

        counter_e = 0
        for i, n in enumerate(list_p[:-1]):
            for j, m in enumerate(list_p[(i+1):]):
                dist = np.linalg.norm(n-m)
                if dist == self.lattice_constant:
                    G.add_edge(i, (i+1)+j, label=counter_e)
                    counter_e += 1

        return G, H

    def set_graph_adjacency(self, V, G, H):

        """
        Return the affiliation list of two dual cubic lattices.

        Args:
            Voronoi (scipy.spatial.Voronoi):\n
                A Voronoi-Tesselation object.
            G (nx.Graph):\n
                The inner networkx graph.
            H (nx.Graph):\n
                The outer networkx graph.

        Return:
            list:\n
                The edge affiliation list of the dual cubic lattice.

        """
        rv_aux = []
        rp_aux = []
        adj = []

        for rv, rp in zip(V.ridge_vertices, V.ridge_points):
            if np.any(np.array(rv) == -1):
                continue
            else:
                rv_aux.append(rv)
                rp_aux.append(rp)

        counter_e = 0

        for i, rv in enumerate(rv_aux):
            E1 = (rp_aux[i][0], rp_aux[i][1])
            for j, v in enumerate(rv):
                e1 = rv[-1+j]
                e2 = rv[-1+(j+1)]
                E2 = (e1, e2)

                if not H.has_edge(*E2):
                    options = {
                        # 'slope': (V.vertices[e1], V.vertices[e2]),
                        'label': counter_e
                    }
                    H.add_edge(*E2, **options)
                    counter_e += 1
                if G.has_edge(*E1):
                    adj.append([G.edges[E1]['label'], H.edges[E2]['label']])

        return adj

@dataclass
class NetworkxDualDiamond(NetworkxDual):

    """
    A class for spatial, dual diamond circuits.

    Attributes
    ----------

        layer (list):\n
            List of the mutlilayered circuits.
        lattice_constant (float):\n
            Scale for the spacing between the networks.
        translation_length (float):\n
            Scale for the translation difference between the multiple networks.

    """
    def __init__(self, num_periods):

        """
        A constructor for multilayer circuit diamond objects, setting default
        values for the interal graph objects and geometry
        """

        super(NetworkxDualDiamond, self).__init__()
        self.lattice_constant = np.sqrt(3.)/2.
        self.translation_length = 1
        self.dualDiamond(num_periods)

    def dualDiamond(self, num_periods):

        """
        Set internal networks structure, dual diamond.

        Args:
            num_periods (int):\n
                Repetition number of the unit cells.

        """

        # create primary point cloud with lattice structure
        adj = []

        ic = NetworkxDiamond(1)
        unit_cell = ic.diamond_unit_cell()

        pg = [0, 0, 0]
        G_aux = self.periodic_cell_structure_offset(unit_cell, num_periods, pg)

        hg = [1, 0, 0]
        H_aux = self.periodic_cell_structure_offset(unit_cell, num_periods, hg)

        G1, G2 = self.init_graph_nuclei(G_aux, H_aux)

        adj = self.set_graph_adjacency(G1, G2)

        # cut off redundant (non-connected or neigborless) points/edges
        G1, G2 = self.prune_leaves(G1, G2, adj)

        # relabeling network nodes/edges & adjacency-list
        P = self.relabel_networkx(G1, G2, adj)

        self.layer = [P[0], P[1]]

    def init_graph_nuclei(self, G_aux, H_aux):
        """
        Generate points for a diamond lattice and its dual via copy+translation
         and return dual graph representations.

        Args:
            G_aux (nx.Graph):\n
                Inner networkx graph.
            H_aux (nx.Graph):\n
                Outer networkx graph.

        Returns:
            nx.Graph:\n
                Inner networkx graph.
            nx.Graph:\n
                Outer networkx graph.

        """

        G = self.init_graph(G_aux)
        H = self.init_graph(H_aux)

        return G, H

    def init_graph(self, G_aux):

        """
        Generate points for a diamond lattice return dual graph
        representations.

        Args:
            G_aux (nx.Graph):\n
                A networkx graph.

        Returns:
            nx.Graph:\n
                A networkx graph.

        """

        G = nx.Graph()
        points_G = [G_aux.nodes[n]['pos'] for i, n in enumerate(G_aux.nodes())]
        counter_e = 0
        counter_n = 0

        for i, n in enumerate(G_aux.nodes()):
            G.add_node(i, pos=G_aux.nodes[n]['pos'], label=counter_n)
            counter_n += 1

        for i, n in enumerate(points_G[:-1]):
            for j, m in enumerate(points_G[(i+1):]):

                dist = np.linalg.norm(np.subtract(n, m))
                if dist == self.lattice_constant:
                    G.add_edge(i, (i+1)+j, label=counter_e)
                    counter_e += 1

        return G


class NetworkxDualLaves(NetworkxDual):

    """
    A class for spatial, dual Laves circuits.

    Attributes
    ----------

        layer (list):\n
            List of the mutlilayered circuits.
        lattice_constant (float):\n
            Scale for the spacing between the networks.

    """
    def __init__(self, num_periods):

        """
        A constructor for multilayer circuit laves objects, setting default
        values for the interal graph objects and geometry
        """

        super(NetworkxDualLaves, self).__init__()
        self.lattice_constant = 2.
        self.dualLaves(num_periods)

    # test new minimal surface graph_sets

    def dualLaves(self, num_periods):

        """
        Set internal networks structure, dual diamond.

        Args:
            num_periods (int):\n
                A networkx graph.

        """

        G_aux = self.laves_graph(num_periods, 'R', [0., 0., 0.])
        H_aux = self.laves_graph(num_periods, 'L', [3., 2., 0.])

        G1, G2 = self.init_graph_nuclei(G_aux, H_aux)

        adj = self.set_graph_adjacency(G1, G2)

        # cut off redundant (non-connected or neigborless) points/edges
        G1, G2 = self.prune_leaves(G1, G2, adj)

        # relabeling network nodes/edges & adjacency-list
        P = self.relabel_networkx(G1, G2, adj)

        self.layer = [P[0], P[1]]

    def laves_graph(self, num_periods, chirality, offset):

        """
        Generate points for a Laves lattice and its dual via mirroring +
        translation and return dual graph representations.

        Args:
            num_periods (int):\n
                Repetition number of the unit cells..
            chirality (string):\n
                Chirality identifier for the current lattice generator.
            offset (ndarray):\n
                A translation vector.


        Returns:
            nx.Graph: A networkx graph

        """

        L = nx.Graph()
        periods = range(num_periods)

        fundamental_points = [
            [0, 0, 0],
            [1, 1, 0],
            [1, 2, 1],
            [0, 3, 1],
            [2, 2, 2],
            [3, 3, 2],
            [3, 0, 3],
            [2, 1, 3]
            ]
        if chirality == 'R':

            for fp in fundamental_points:
                for i in periods:
                    for j in periods:
                        for k in periods:

                            pos_n = np.add(
                                np.add(fp, [4.*i, 4.*j, 4.*k]), offset
                                )
                            L.add_node(tuple(pos_n), pos=pos_n)
        if chirality == 'L':
            for fp in fundamental_points:
                for i in periods:
                    for j in periods:
                        for k in periods:

                            pos_n = np.add(
                                np.add(
                                    np.multiply(fp, [-1., 1., 1.]),
                                    [4.*i, 4.*j, 4.*k]
                                ),
                                offset
                                )
                            L.add_node(tuple(pos_n), pos=pos_n)

        return L

    def init_graph_nuclei(self, G_aux, H_aux):

        """
        Generate points for a Laves lattice and its dual via mirroring +
        translation and return dual graph representations.

        Args:
            G_aux (nx.Graph):\n
                Inner networkx graph.
            H_aux (nx.Graph):\n
                Outer networkx graph.

        Returns:
            nx.Graph: A networkx graph

        """

        G = self.init_graph(G_aux)
        H = self.init_graph(H_aux)

        return G, H

    def init_graph(self, G_aux):

        """
        Built labeled, attributed graphs from raw point sets.

        Args:
            G_aux (nx.Graph):\n
                Inner networkx graph holding unlabeled data.

        Returns:
            nx.Graph: A networkx graph

        """

        list_nodes = list(G_aux.nodes())

        G = nx.Graph()
        counter_e = 0
        counter_n = 0
        for i, n in enumerate(G_aux.nodes()):
            G.add_node(n, pos=G_aux.nodes[n]['pos'], label=counter_n)
            counter_n += 1

        for i, n in enumerate(list_nodes[:-1]):
            for j, m in enumerate(list_nodes[(i+1):]):

                v = np.subtract(n, m)
                dist = np.dot(v, v)

                if dist == self.lattice_constant:
                    options = {
                        # 'slope': (G_aux.nodes[n]['pos'],
                        # G_aux.nodes[m]['pos']),
                        'label': counter_e
                    }
                    G.add_edge(n, m, **options)
                    counter_e += 1

        return G


class NetworkxDualCatenation(NetworkxDual):

    """
    A class for spatial, dual Laves circuits.

    Attributes
    ----------

        layer (list):\n
            List of the mutlilayered circuits.
        lattice_constant (float):\n
            Scale for the spacing between the networks.

    """
    def __init__(self, num_periods):

        """
        A constructor for multilayer circuit catenation objects, setting
        default values for the interal graph objects and geometry.
        """

        super(NetworkxDualCatenation, self).__init__()
        self.lattice_constant = 1
        self.translation_length = 1
        self.dual_ladder(num_periods)

    def dual_ladder(self, num_periods):

        """
        Set internal networkx structure, dual ladder.

        Args:
            num_periods (int):\n
                Repetition number of the unit tile.

        """

        np1 = [num_periods, 1]
        np2 = [num_periods+1, 1]

        N = [np1, np2]
        ic = NetworkxSquare
        G1, G2 = [nx.Graph(ic(i).G) for i in N]

        theta = np.pi/2.
        rot_mat = np.array(
                (
                    (1, 0, 0),
                    (0, np.cos(theta), -np.sin(theta)),
                    (0, np.sin(theta), np.cos(theta))
                )
        )

        for n in G1.nodes():
            p = np.array(G1.nodes[n]['pos'])
            p1 = np.dot(rot_mat, p)
            p2 = np.add([0.5, 0.5, -0.5], p1)

            G1.nodes[n]['pos'] = self.lattice_constant*p2

        self.layer = [G1, G2]


class NetworkxDualCrossMesh(NetworkxDual):

    """
    A class for spatial, dual Laves circuits.

    Attributes
    ----------

        layer (list):\n
            List of the mutlilayered circuits.
        lattice_constant (float):\n
            Scale for the spacing between the networks.

    """
    def __init__(self, num_periods):

        """
        A constructor for multilayer circuit catenation objects, setting
        default values for the interal graph objects and geometry
        """

        super(NetworkxDualCrossMesh, self).__init__()
        self.lattice_constant = 1
        self.translation_length = 1
        self.dualCrossMesh(num_periods)

    def dualCrossMesh(self, n_periods):

        """
        Set internal networkx structure, dual crossed_mesh.

        Args:
            num_periods (int):\n
                Repetition number of the unit tile.

        """
        num_periods1X, num_periods1Y, num_periods2X, num_periods2Y = n_periods

        np1 = [num_periods1X, num_periods1Y]
        np2 = [num_periods2X, num_periods2Y]

        N = [np1, np2]
        ic = NetworkxSquare
        G1, G2 = [nx.Graph(ic(i).G) for i in N]

        theta = np.pi/2.
        rot_mat = np.array(
            (
                (1, 0, 0),
                (0, np.cos(theta), -np.sin(theta)),
                (0, np.sin(theta), np.cos(theta))
                )
            )

        for n in G1.nodes():
            p = np.array(G1.nodes[n]['pos'])
            p1 = np.dot(rot_mat, p)
            p2 = np.add([0.5, 0.5, -0.5], p1)

            G1.nodes[n]['pos'] = self.lattice_constant*p2

        self.layer = [G1, G2]

In [ ]:
###### Data class ######
import sys
import pandas as pd

@dataclass
class Circuit:

    """
    A class of linear circuits (for lumped parameter modelling).

    Attributes
    ----------
        scales (dictionary):\n
            A dictionary holding the unit system.
        graph (dictionary):\n
            A dictionary holding circuit initial conditions.
        nodes (pd.DataFrame):\n
            A container for node data.
        edges (pd.DataFrame):\n
            A container for edge data.
        draw_weight_scaling (float):\n
            Standard wights for drawing.

    """

    G: nx.Graph = field(default_factory=nx.Graph, repr=False)
    info: str = 'unknown'
    scales: dict = field(default_factory=dict, repr=False, init=False)
    graph: dict = field(default_factory=dict, repr=False, init=False)
    nodes: pd.DataFrame = field(
                            default_factory=pd.DataFrame,
                            repr=False,
                            init=False
                        )
    edges: pd.DataFrame = field(
                            default_factory=pd.DataFrame,
                            repr=False,
                            init=False
                        )

    def __post_init__(self):

        self.init_circuit()

    def set_graph_containers(self):

        """
        Set internal graph containers.

        """

        self.H = nx.Graph()
        self.H_C = []
        self.H_J = []

    def set_info(self, input_graph, grid_type):

        e = nx.number_of_edges(input_graph)
        n = nx.number_of_nodes(input_graph)

        return f'type: {grid_type}, #edges: {e}, #nodes: {n}'

    def init_circuit(self):

        self.info = self.set_info(self.G, 'custom')

        self.scales = {
                'conductance': 1,
                'flow': 1,
                'length': 1
            }

        self.graph = {
            'source_mode': '',
            'plexus_mode': '',
            'threshold': 0.001,
            'num_sources': 1
        }

        self.nodes = pd.DataFrame(
            {
                'source': [],
                'potential': [],
                'label': [],
            }
        )

        self.edges = pd.DataFrame(
            {
                'conductivity': [],
                'flow_rate': [],
                'label': [],
            }
        )
        self.set_graph_containers()
        self.default_init()

    def default_init(self):

        """
        Initialize the default setting of a circuit, by taking a networkx graph
        and setting containers

        """
        self.draw_weight_scaling = 1.

        options = {
            'first_label': 0,
            'ordering': 'default'
            }
        self.G = nx.convert_node_labels_to_integers(self.G, **options)
        # self.initialize_circuit()

        self.list_graph_nodes = list(self.G.nodes())
        self.list_graph_edges = list(self.G.edges())

        init_val = ['#269ab3', 0, 0, 5]
        init_attributes = ['color', 'source', 'potential', 'conductivity']

        for i, val in enumerate(init_val):
            nx.set_node_attributes(self.G, val, name=init_attributes[i])

        E = self.G.number_of_edges()
        N = self.G.number_of_nodes()

        for k in self.nodes:
            if k == 'label':
                self.nodes[k] = [n for n in self.list_graph_nodes]
            else:
                self.nodes[k] = np.zeros(N)

        for k in self.edges:
            if k == 'label':
                self.edges[k] = [e for e in self.list_graph_edges]
            else:
                self.edges[k] = np.zeros(E)

    def get_incidence_matrices(self):

        """
        Get the incidence matrices from the internal graph objects.

        Returns:
            ndarray:\n
                A internal circuit graph's incidence matrix.
            ndarray.T:\n
                A internal circuit graph's incidence matrix, transposed.

        """

        options = {
            'nodelist': self.list_graph_nodes,
            'edgelist': self.list_graph_edges,
            'oriented': True,
        }

        B = nx.incidence_matrix(self.G, **options).toarray()
        BT = np.transpose(B)

        return B, BT

    # update network traits from dynamic data
    def set_network_attributes(self):
        """
        Set the internal DataFrames with the current graph state.
        """

        # set potential node values
        for i, n in enumerate(self.list_graph_nodes):

            self.G.nodes[n]['potential'] = self.nodes['potential'][i]
            self.G.nodes[n]['label'] = i
        # set conductivity matrix
        for j, e in enumerate(self.list_graph_edges):
            self.G.edges[e]['conductivity'] = self.edges['conductivity'][j]
            self.G.edges[e]['label'] = j

    # clipp small edges & translate conductance into general edge weight
    def clipp_graph(self):

        """
        Prune the internal graph and generate a new internal variable
        represting the pruned based on an interanl threshold value.

        """

        # cut out edges which lie beneath a certain threshold value and export
        # this clipped structure
        self.set_network_attributes()
        self.threshold = 0.01

        for e in self.list_graph_edges:
            if self.G.edges[e]['conductivity'] > self.threshold:
                self.H.add_edge(*e)
                for k in self.G.edges[e].keys():
                    self.H.edges[e][k] = self.G.edges[e][k]

        list_pruned_nodes = list(self.H.nodes())
        list_pruned_edges = list(self.H.edges())

        for n in list_pruned_nodes:
            for k in self.G.nodes[n].keys():
                self.H.nodes[n][k] = self.G.nodes[n][k]
            self.H_J.append(self.G.nodes[n]['source'])
        for e in list_pruned_edges:
            self.H_C.append(self.H.edges[e]['conductivity'])

        self.H_C = np.asarray(self.H_C)
        self.H_J = np.asarray(self.H_J)

        assert(len(list(self.H.nodes())) == 0)

    def calc_root_incidence(self):

        """
        Find the incidence for a system with binary-type periphehal nodes.

        Returns:
            list:\n
                A list of nodes adjacent to the source.
            list:\n
                A list of nodes adjacent to the sink.

        """

        root = 0
        sink = 0

        for i, n in enumerate(self.list_graph_nodes):
            if self.G.nodes[n]['source'] > 0:
                root = n
            if self.G.nodes[n]['source'] < 0:
                sink = n

        E_1 = list(self.G.edges(root))
        E_2 = list(self.G.edges(sink))
        E_ROOT = []
        E_SINK = []
        for e in E_1:
            if e[0] != root:
                E_ROOT += list(self.G.edges(e[0]))
            else:
                E_ROOT += list(self.G.edges(e[1]))

        for e in E_2:
            if e[0] != sink:
                E_SINK += list(self.G.edges(e[0]))
            else:
                E_SINK += list(self.edges(e[1]))

        return E_ROOT, E_SINK

    # test consistency of conductancies & sources
    def test_source_consistency(self):

        """
        Test whether boundaries conditions for sources on the internal graph
        variable are consistenly set.

        """

        self.set_network_attributes()
        tolerance = 0.00001
        # check value consistency
        S = nx.get_node_attributes(self.G, 'source').values()
        sources = np.fromiter(S, float)

        assert(np.sum(sources) < tolerance)

        A1 = 'set_source_landscape(): '
        A2 = ' is set and consistent :)'
        print(A1+self.graph['source_mode']+A2)

    def test_conductance_consistency(self):
        """
        Test whether boundaries conditions for edge consuctancies on the
        internal graph variable are consistenly set.

        """
        self.set_network_attributes()

        # check value consistency
        K = nx.get_edge_attributes(self.G, 'conductivity').values()
        conductivities = np.fromiter(K, float)

        assert(len(np.where(conductivities <= 0)[0]) == 0)

        A1 = 'set_plexus_landscape(): '
        A2 = ' is set and consistent :)'
        print(A1+self.graph['plexus_mode']+A2)

    def get_pos(self):
        """
        Getting positions of the vertices from the internal graphs.

        Returns:
            dictionary:\n
                A dictionary holding nodes and their positions in euclidean
                space.

        """

        pos_key = 'pos'
        reset_layout = False
        for j, n in enumerate(self.G.nodes()):
            if pos_key not in self.G.nodes[n]:
                reset_layout = True
        if reset_layout:
            print('set networkx.spring_layout()')
            pos = nx.spring_layout(self.G)
        else:
            pos = nx.get_node_attributes(self.G, 'pos')

        return pos

    def set_pos(self, pos_data={}):

        """
        Set the postions of the internal graph.

        Args:
            pos_data (dictionary):\n
                A dictionary of nodal positions.

        """

        pos_key = 'pos'
        reset_layout = False
        nodata = False
        if len(pos_data.values()) == 0:
            nodata = True

        for j, n in enumerate(self.G.nodes()):
            if pos_key not in self.G.nodes[n]:
                reset_layout = True
        if reset_layout and nodata:
            print('set networkx.spring_layout()')
            pos = nx.spring_layout(self.G)
            nx.set_node_attributes(self.G, pos, 'pos')
        else:
            nx.set_node_attributes(self.G, pos_data, 'pos')

    def set_scale_pars(self, new_parameters):
        """
        Set a new internal unit system.

        Args:
            new_parameters (dictionary):\n
                A new set of units to bet set.

        """

        self.scales = new_parameters

    def set_graph_pars(self, new_parameters):
        """
        Set a circuit boundary conditions.

        Args:
            new_parameters (dictionary):\n
                A new set of conditions to bet set.

        """
        self.graph = new_parameters

    # output
    def plot_circuit(self, *args, **kwargs):

        """
        Use Plotly.GraphObjects to create interactive plots that have
        optionally the graph atributes displayed.

        Args:
            args (list):\n
                A list of keywords for the internal edge and nodal DataFrames
                which are to be displayed.
            kwargs (dictionary):\n
                A dictionary for plotly keywords customizing the plots' layout.

        Returns:
            GraphObject.Figure: A plotly figure displaying the circuit.

        """

        self.set_pos()
        E = self.get_edges_data(*args)
        V = self.get_nodes_data(*args)

        options = {
            'edge_list': self.list_graph_edges,
            'node_list': self.list_graph_nodes,
            'edge_data': E,
            'node_data': V
        }

        if type(kwargs) is not None:
            options.update(kwargs)

        fig = plot_networkx(self.G, **options)

        return fig

    def get_nodes_data(self, *args):
        """
        Get internal nodal DataFrame columns by keywords.

        Args:
            args (list):\n
                A list of keywords to check for in the internal DataFrames.

        Returns:
            pd.DataFrame: A cliced DataFrame.

        Raises:
            Exception: description

        """

        cols = ['label']
        cols += [a for a in args if a in self.nodes.columns]

        dn = pd.DataFrame(self.nodes[cols])

        return dn

    def get_edges_data(self, *args):

        """
        Get internal nodal DataFrame columns by keywords.

        Args:
            args (list):\n
                A list of keywords to check for in the internal DataFrames.

        Returns:
            pd.DataFrame: A cliced DataFrame.

        Raises:
            Exception: description

        """
        cols = ['label']
        cols += [a for a in args if a in self.edges.columns]

        de = pd.DataFrame(self.edges[cols])

        return de

@dataclass
class FlowCircuit(Circuit):

    """
    A derived class for flow circuits.

    Attributes
    ----------

        source_mode (dictionary):\n
            A dictionary of custom source-sink boundaries.
        plexus_mode (dictionary):\n
            A dictionary of custom plexus initializations.

    """

    source_mode: dict = field(default_factory=dict, repr=False)
    plexus_mode: dict = field(default_factory=dict, repr=False)

    def __post_init__(self):

        self.init_circuit()
        self.set_flowModes()

    def set_flowModes(self):

        self.source_mode = {
            'default': self.init_source_default,
            'root_geometric': self.init_source_root_central_geometric,
            'root_short': self.init_source_root_short,
            'root_long': self.init_source_root_long,
            'dipole_border': self.init_source_dipole_border,
            'dipole_wall': self.init_source_dipole_wall,
            'dipole_point': self.init_source_dipole_point,
            'root_multi': self.init_source_root_multi,
            'custom': self.init_source_custom
        }

        self.plexus_mode = {
            'default': self.init_plexus_default,
            'custom': self.init_plexus_custom,
        }

    # set a certain set of boundary conditions for the given networks
    def set_source_landscape(self, modeSRC='default', **kwargs):

        """
        Set the internal bounday state of sinks and sources.

        Args:
            mode (string): The specific source mode.
            kwargs (dictonary): Source attribute specifiers, optional.

        """

        # optional keywords
        if 'num_sources' in kwargs:
            self.graph['num_sources'] = kwargs['num_sources']

        elif 'sources' in kwargs:
            self.custom = kwargs['sources']

        # else:
        #     print('Warning: Not recognizing certain keywords')
        # call init sources
        if modeSRC in self.source_mode.keys():
            print(f'Set source: {modeSRC}')
            self.source_mode[modeSRC]()

        else:
            sys.exit(
                'Whooops, Error: Define Input/output-flows for the network.'
                )

        self.graph['source_mode'] = modeSRC
        self.test_source_consistency()

    def set_potential_landscape(self, mode):

        # todo
        return 0

    # different init source functions
    def init_source_custom(self):

        """
        Set custom sinks and sources boundaries.
        """

        if len(self.custom.keys()) == len(self.list_graph_nodes):

            for j, node in enumerate(self.list_graph_nodes):

                s = self.custom[node]*self.scales['flow']
                self.G.nodes[node]['source'] = s
                self.nodes.at[j, 'source'] = s

        else:
            print(
                'Warning, custom source values ill defined, setting default!'
                )
            self.init_source_default()

    def init_source_default(self):

        """
        Set one topologically central source, sinks otherwise.
        """

        centrality = nx.betweenness_centrality(self.G)
        centrality_sorted = sorted(centrality, key=centrality.__getitem__)

        self.set_root_leaves_relationship(centrality_sorted[-1])

    def init_source_root_central_geometric(self):

        """
        Set one geometrically central source, sinks otherwise.
        """

        pos = self.get_pos()
        X = np.mean(list(pos.values()), axis=0)

        dist = {}
        for n in self.list_graph_nodes:
            dist[n] = np.linalg.norm(np.subtract(X, pos[n]))
        sorted_dist = sorted(dist, key=dist.__getitem__)

        self.set_root_leaves_relationship(sorted_dist[0])

    def init_source_root_short(self):

        """
        Set one-sided source (right), sinks otherwise.
        """

        # check whether geometric layout has been set
        pos = self.get_pos()

        # check for root closests to coordinate origin
        dist = {}
        for n, p in pos.items():
            dist[n] = np.linalg.norm(p)
        sorted_dist = sorted(dist, key=dist.__getitem__)

        self.set_root_leaves_relationship(sorted_dist[0])

    def init_source_root_long(self):

        """
        Set one-sided source (left), sinks otherwise.
        """

        # check whether geometric layout has been set
        pos = self.get_pos()

        # check for root closests to coordinate origin
        dist = {}
        for n, p in pos.items():
            dist[n] = np.linalg.norm(p)
        sorted_dist = sorted(dist, key=dist.__getitem__, reverse=True)

        self.set_root_leaves_relationship(sorted_dist[0])

    def init_source_dipole_border(self):

        """
        Set sources on one side of the graph, sinks on the opposing side.
        """

        pos = self.get_pos()
        dist = {}
        for n, p in pos.items():
            dist[n] = np.linalg.norm(p[0])

        vals = list(dist.values())
        max_x = np.amax(vals)
        min_x = np.amin(vals)

        max_idx = []
        min_idx = []
        for k, v in dist.items():
            if v == max_x:
                max_idx.append(k)

            elif v == min_x:
                min_idx.append(k)

        self.set_poles_relationship(max_idx, min_idx)

    def init_source_dipole_wall(self):

        """
        Set sources on one side of the graph, sinks on the opposing side.
        """

        pos = self.get_pos()
        dist = {}
        for n, p in pos.items():
            dist[n] = np.linalg.norm(p[0])

        vals = list(dist.values())
        max_x = np.amax(vals)

        max_idx = []
        min_idx = []
        for k, v in dist.items():
            if v == max_x:
                max_idx.append(k)

            else:
                min_idx.append(k)

        self.set_poles_relationship(max_idx, min_idx)

    def init_source_dipole_point(self):

        """
        Set a single source-sink pair.
        """

        dist = {}
        for j, n in enumerate(self.list_graph_nodes[:-2]):
            for i, m in enumerate(self.list_graph_nodes[j+1:]):
                path = nx.shortest_path(self.G, source=n, target=m)
                dist[(n, m)] = len(path)
        max_len = np.amax(list(dist.values()))
        push = []
        for key in dist.keys():
            if dist[key] == max_len:
                push.append(key)

        idx = np.random.choice(range(len(push)))
        source, sink = push[idx]

        self.set_poles_relationship([source], [sink])

    def init_source_root_multi(self):

        """
        Set multiple random sources, sinks otherwise.
        """

        idx = np.random.choice(
            self.list_graph_nodes, size=self.graph['num_sources']
            )
        self.nodes_source = [
                self.G.number_of_nodes()/self.graph['num_sources']-1,
                -1
                ]

        for j, n in enumerate(self.list_graph_nodes):

            if n in idx:

                self.set_source_attributes(j, n, 0)

            else:

                self.set_source_attributes(j, n, 1)

    # auxillary function for the block above
    def set_root_leaves_relationship(self, root):

        """
        Set boundaries with distinguished root vertex.
        """

        self.nodes_source = [self.G.number_of_nodes()-1, -1]
        for j, n in enumerate(self.list_graph_nodes):

            if n == root:
                idx = 0

            else:
                idx = 1

            self.set_source_attributes(j, n, idx)

    def set_poles_relationship(self, sources, sinks):

        """
        Set boundaries with distinguished pole vertices.
        """

        self.nodes_source = [1, -1, 0]

        # neutral
        for j, n in enumerate(self.list_graph_nodes):
            self.set_source_attributes(j, n, 2)

        # sources
        for i, s in enumerate(sources):
            for j, n in enumerate(self.list_graph_nodes):
                if n == s:
                    self.set_source_attributes(j, s, 0)

        # sinks
        for i, s in enumerate(sinks):
            for j, n in enumerate(self.list_graph_nodes):
                if n == s:
                    self.set_source_attributes(j, s, 1)

    def set_source_attributes(self, j, node, idx):

        """
        Set nodal flow attributes.
        """
        val = self.nodes_source[idx]*self.scales['flow']
        self.G.nodes[node]['source'] = val
        self.nodes.at[j, 'source'] = val

    # different init potetnial functions
    def set_terminals_potentials(self, p0):

        """
        Set nodal potentials.
        """

        idx_potential = []
        idx_sources = []
        for j, n in enumerate(nx.nodes(self.G)):

            if self.G.nodes[n]['source'] > 0:
                self.G.nodes[n]['potential'] = 1
                self.V[j] = p0
                idx_potential.append(j)
            elif self.G.nodes[n]['source'] < 0:

                self.G.nodes[n]['potential'] = 0.
                self.V[j] = 0.
                idx_potential.append(j)
            else:
                self.G.nodes[n]['source'] = 0.
                self.J[j] = 0.
                idx_sources.append(j)

        self.G.graph['sources'] = idx_sources
        self.G.graph['potentials'] = idx_potential

    # different init plexus functions
    def set_plexus_landscape(self, modePLX='default', **kwargs):

        """
        Set the intial conductivity landscape of the plexus.

        Args:
            mode (string): The specific plexus mode.
            kwargs (dictonary): Plexus attribute specifiers, optional.
        """

        # optional keywords

        if 'plexus' in kwargs:

            self.custom = kwargs['plexus']

        # call init sources
        if modePLX in self.plexus_mode.keys():
            print(f'Set plexus: {modePLX}')
            self.plexus_mode[modePLX]()

        else:
            sys.exit(
                'Whooops, Error: Define proper conductancies for the network.'
                )

        self.graph['plexus_mode'] = modePLX
        self.test_conductance_consistency()

    def init_plexus_default(self):

        """
        Set random initial plexus.
        """

        # find magnitude of flows and set scale of for conductancies
        d = np.amax(self.nodes['source']) * 0.5
        m = self.G.number_of_edges()

        displaced = np.add(np.ones(m), np.random.rand(m))
        self.edges['conductivity'] = np.multiply(d, displaced)

    def init_plexus_custom(self):

        """
        Set customized initial plexus.
        """

        if len(self.custom.keys()) == len(self.list_graph_edges):
            # find magnitude of flows and set scale of for conductancies
            for j, edge in enumerate(self.list_graph_edges):

                c = self.custom[edge]*self.scales['conductance']
                self.G.edges[edge]['conductivity'] = c
                self.edges['conductivity'][j] = c
        else:

            print(
                '''
                Warning, custom conductance values ill defined, setting
                default!
                '''
                )
            self.init_plexus_default()

    # output with draw_netowrkx
    def get_nodes_data(self, *args):

        """
        Get internal nodal DataFrame columns by keywords.

        Args:
            args (list):\n
                A list of keywords to check for in the internal DataFrames.

        Returns:
            pd.DataFrame: A cliced DataFrame.

        """

        cols = ['label', 'source']
        cols += [a for a in args if a in self.nodes.columns]

        dn = pd.DataFrame(self.nodes[cols])

        return dn

    def get_edges_data(self, *args):

        """
        Get internal nodal DataFrame columns by keywords.

        Args:
            args (list):\n
                A list of keywords to check for in the internal DataFrames.

        Returns:
            pd.DataFrame: A cliced DataFrame.

        """

        cols = ['label', 'conductivity', 'flow_rate']
        cols += [a for a in args if a in self.edges.columns]

        de = pd.DataFrame(self.edges[cols])

        return de

    def plot_circuit(self, *args, **kwargs):

        """
        Use Plotly.GraphObjects to create interactive plots that have
        optionally the graph atributes displayed.

        Args:
            args (list):\n
                A list of keywords for the internal edge and nodal DataFrames
                which are to be displayed.
            kwargs (dictionary):\n
                A dictionary for plotly keywords customizing the plots' layout.

        Returns:
            plotly.graph_objects.Figure: A plotly figure displaying the
            circuit.

        """

        self.set_pos()
        E = self.get_edges_data(*args)
        V = self.get_nodes_data(*args)

        opt = {
            'edge_list': self.list_graph_edges,
            'node_list': self.list_graph_nodes,
            'edge_data': E,
            'node_data': V
        }
        if type(kwargs) is not None:
            opt.update(kwargs)

        fig = plot_networkx(self.G, **opt)

        return fig

@dataclass
class FluxCircuit(FlowCircuit):

    """
    A derived class for flux circuits.

    Attributes
    ----------

        solute_mode (dictionary):\n
            A dictionary of custom solute outflux/influx boundaries.
        absorption_mode (dictionary):\n
            A dictionary of custom absorption rate initializations.
        geom_mode (dictionary):\n
            A dictionary of custom plexus geometrical initializations.

    """

    def __post_init__(self):

        self.init_circuit()
        self.set_flowModes()

        e = self.G.number_of_edges()
        n = self.G.number_of_nodes()

        newNodeAttr = ['solute', 'concentration']
        for k in newNodeAttr:
            self.nodes[k] = np.zeros(n)

        newEdgeAttr = [
            'peclet',
            'absorption',
            'length',
            'radius',
            'radius_sq',
            'uptake'
            ]
        for k in newEdgeAttr:
            self.edges[k] = np.zeros(e)

        newScaleAttr = ['flux', 'sum_flux', 'diffusion', 'absorption']
        for k in newScaleAttr:
            self.scales.update({k: 1})

        newGraphAttr = [
            'solute_mode',
            'absorption_mode',
            'geom_mode'
            ]

        for k in newGraphAttr:
            self.graph.update({k: ''})

        self.solute_mode = {
            'default': self.init_solute_default,
            'custom': self.init_solute_custom
        }

        self.absorption_mode = {
            'default': self.init_absorption_default,
            'random': self.init_absorption_random,
            'custom': self.init_absorption_custom
        }

        self.geom_mode = {
            'default': self.init_geom_default,
            'random': self.init_geom_random,
            'custom': self.init_geom_custom
        }

    # set injection and outlet of metabolites
    def set_solute_landscape(self, mode='default', **kwargs):

        """
        Set the internal bounday state of sinks and sources.

        Args:
            mode (string):\n
                The specific solute mode.
            kwargs (dictonary):\n
                Solute attribute specifiers, optional.

        """

        # optional keywords
        if 'solute' in kwargs:
            self.custom = kwargs['solute']

        # call init sources
        if mode in self.solute_mode.keys():

            self.solute_mode[mode]()

        else:
            sys.exit(
                'Whooops, Error: Define Input/output-flows for the network.'
                )

        self.graph['solute_mode'] = mode

    def init_solute_default(self):

        """
        Set solute out/influx accordingly to the sinks-sources boundaries.
        """

        vals = [1, -1, 0]

        for j, n in enumerate(self.list_graph_nodes):

            if self.nodes['source'][j] > 0:
                self.set_solute(j, n, vals[0])

            elif self.nodes['source'][j] < 0:
                self.set_solute(j, n, vals[1])

            else:
                self.set_solute(j, n, vals[2])

    def init_solute_custom(self):

        """
        Set custom solute out/influx boundaries.
        """

        if len(self.custom.keys()) == len(self.list_graph_nodes):

            for j, node in enumerate(self.list_graph_nodes):

                f = self.custom[node]*self.scales['flux']
                self.nodes.at[j, 'solute'] = f
                self.G.nodes[node]['solute'] = f

        else:

            print(
                'Warning, custom solute values ill defined, setting default!'
                )
            self.init_solute_default()

    def set_solute(self, idx, nodes, vals):

        """
        Set nodal solute in/outflux attributes.

        Args:
            idx (int): The dataframe vertex indicator.
            nodes (int): The corresponding networkx graph node.
            vals (float): Solute value to be set.
        """

        f = self.scales['flux']*vals
        self.nodes.at[idx, 'solute'] = f
        self.G.nodes[nodes]['solute'] = f

    # set spatial pattern of solute absorption rate
    def set_absorption_landscape(self, mode='default', **kwargs):

        """
        Set the internal bounday state of sinks and sources.

        Args:
            mode (string):\n
                The specific absorption mode.
            kwargs (dictonary):\n
                Absorption attribute specifiers, optional.

        """

        # optional keywords
        if 'absorption' in kwargs:
            self.custom = kwargs['absorption']

        # call init sources
        if mode in self.absorption_mode.keys():

            self.absorption_mode[mode]()

        else:
            sys.exit(
                '''
                Whooops, Error: Define absorption rate pattern for the network.
                '''
                )

        self.graph['absorption_mode'] = mode

    def init_absorption_default(self):
        """
        Set a constant absorption rate landscape.
        """

        num_e = self.G.number_of_edges()
        self.edges['absorption'] = np.ones(num_e)*self.scales['absorption']

    def init_absorption_random(self):
        """
        Set a random absorption rate landscape.
        """

        num_e = self.G.number_of_edges()
        rnd = np.random.rand(num_e)
        self.edges['absorption'] = rnd*self.scales['absorption']

    def init_absorption_custom(self):

        """
        Set a custom absorption rate landscape.
        """

        if len(self.custom.keys()) == len(self.list_graph_edges):
            for j, edge in enumerate(self.list_graph_edges):

                c = self.custom[edge]*self.scales['absorption']
                self.edges['absorption'][j] = c
        else:

            print(
                '''
                Warning, custom absorption values ill defined, setting default!
                '''
                )
            self.init_absorption_default()

    # set spatial pattern of length and radii
    def set_geom_landscape(self, mode='default', **kwargs):

        """
        Set the internal bounday state of sinks and sources.

        Args:
            mode (string):\n
                The specific geometric initialization mode.
            kwargs (dictonary):\n
                Geometric initialization specifiers, optional.

        """

        # optional keywords
        if 'geom' in kwargs:
            self.custom = kwargs['geom']

        # call init sources
        if mode in self.geom_mode.keys():

            self.geom_mode[mode]()

        else:
            sys.exit(
                'Whooops, Error: Define micro geometrics for the network.'
                )

        self.graph['geom_mode'] = mode

    def init_geom_default(self):

        """
        Set a constant length for all connections.
        """

        num_e = self.G.number_of_edges()
        self.edges['length'] = np.ones(num_e)*self.scales['length']
        conductivity = self.edges['conductivity']
        k = self.scales['conductance']
        self.edges['radius'] = np.power(conductivity/k, 0.25)
        self.edges['radius_sq'] = np.sqrt(conductivity/k)

    def init_geom_random(self):

        """
        Set a random length for all connections.
        """

        num_e = self.G.number_of_edges()
        self.edges['length'] = np.random.rand(num_e)*self.scales['length']

    def init_geom_custom(self, flux):

        """
        Set a custom length for all connections.
        """

        if len(self.custom.keys()) == len(self.list_graph_edges):
            for j, edge in enumerate(self.list_graph_edges):

                c = self.custom[edge]*self.scales['length']
                self.edges['length'][j] = c
        else:
            print(
                '''
                Warning, custom absorption values ill defined, setting default!
                '''
                )
            self.init_geom_default()

    def get_nodes_data(self):

        """
        Get internal nodal DataFrame columns by keywords.

        Args:
            args (list):\n
                A list of keywords to check for in the internal DataFrames.

        Returns:
            pd.DataFrame: A cliced DataFrame.

        Raises:
            Exception: description

        """

        dn = pd.DataFrame(self.nodes[['source', 'solute', 'concentration']])

        return dn

    def get_edges_data(self, **kwargs):

        """
        Get internal nodal DataFrame columns by keywords.

        Args:
            args (list):\n
                A list of keywords to check for in the internal DataFrames.

        Returns:
            pd.DataFrame: A cliced DataFrame.

        Raises:
            Exception: description

        """

        de = pd.DataFrame(
            self.edges[
                [
                    'conductivity',
                    'flow_rate',
                    'absorption',
                    'uptake',
                    'peclet',
                    'length'
                    ]
                ]
            )

        if 'width' in kwargs:
            w = np.absolute(self.edges[kwargs['width']].to_numpy())
            de['weight'] = w*self.draw_weight_scaling
        else:
            w = np.power(self.edges['conductivity'].to_numpy(), 0.25)
            de['weight'] = w*self.draw_weight_scaling

        return de

@dataclass
class DualCircuit():

    """
    A base class for flow circuits.

    Attributes
    ----------
        layer (list):\n
            List of the graphs contained in the multilayer circuit.
        e_adj (list):\n
            A list off edge affiliation between the different layers, edge
            view.
        e_adj_idx (list):\n
            A list off edge affiliation between the different layers, label
            view.
        n_adj (list): An internal nodal varaible.
    """

    layer: list = field(default_factory=list, repr=False)
    e_adj: list = field(default_factory=list, repr=False)
    e_adj_idx: list = field(default_factory=list, repr=False)
    n_adj: list = field(default_factory=list, repr=False)

    def distance_edges(self):

        """
        Compute the distance of affiliated edges in the multilayer circuit.

        """

        self.dist_adj = np.zeros(len(self.e_adj_idx))

        for i, e in enumerate(self.e_adj_idx):

            g1 = self.layer[0].G
            pos1 = [g1.nodes[node]['pos'] for node in e[0]]
            p1 = np.mean(pos1, axis=0)

            g2 = self.layer[1].G
            pos2 = [g2.nodes[node]['pos'] for node in e[1]]
            p2 = np.mean(pos2, axis=0)

            self.dist_adj[i] = np.linalg.norm(p1-p2)

    # def check_no_overlap(self, scale):
    #
    #     """
    #     Test whether the multilayer systems have geometrically
    # overlapping edges.
    #
    #     Returns:
    #         bool: True if systems geometrically overlap, otherwise False
    #
    #     """
    #
    #
    #     check = True
    #     K1 = self.layer[0]
    #     K2 = self.layer[1]
    #
    #     for e in self.e_adj:
    #         r1 = K1.C[e[0], e[0]]
    #         r2 = K2.C[e[1], e[1]]
    #
    #         if r1+r2 > scale*0.5:
    #             check = False
    #             break
    #
    #     return check

    # def clipp_graph(self):
    #
    #     """
    #     Prune the internal graph variables, using an edge weight threshold
    # criterium.
    #     """
    #
    #     for i in range(2):
    #         self.layer[i].clipp_graph()

    # output
    def plot_circuit(self, *args, **kwargs):

        """
        Use Plotly.GraphObjects to create interactive plots that have
        optionally the graph atributes displayed.
        Args:
            kwargs (dictionary):\n
                A dictionary for plotly keywords customizing the plots' layout.

        Returns:
            plotly.graph_objects.Figure: A plotly figure displaying the
            circuit.

        """
        fig = plot_networkx_dual(self, *args, **kwargs)

        return fig

In [ ]:
###### Create circuts ######
def init_dual_minsurf_graphs(dual_type, num_periods):

    """
    Initialize a dual spatially embedded multilayer graph, with internal graphs
    based on the network skeletons of triply-periodic minimal surfaces.

    Args:
        dual_type (string):\n
            The type of dual skeleton (simple, diamond, laves, catenation).
        num_periods (int):\n
            Repetition number of the lattice's unit cell.

    Returns:
        NetworkxDual: A dual networkx object.

    """

    plexus_mode = {
        'default': NetworkxDualSimple,
        'simple': NetworkxDualSimple,
        'diamond': NetworkxDualDiamond,
        'laves': NetworkxDualLaves,
    }

    if dual_type in plexus_mode:
        dual_graph = plexus_mode[dual_type](num_periods)

    else:
        print('Warning: Invalid graph mode, choose default')
        dual_graph = plexus_mode['default'](num_periods)

    return dual_graph

def init_dualCatenation(dual_type, num_periods):

    """
    Initialize a dual spatially embedded multilayer graph, with internal graphs
    based on simple catenated network skeletons.

    Args:
        dual_type (string):\n
            The type of dual skeleton (simple, diamond, laves, catenation).
        num_periods (int):\n
            Repetition number of the lattice's unit cell.

    Returns:
        NetworkxDual: A dual networkx object.

    """
    plexus_mode = {
        'default': NetworkxDualCatenation,
        'catenation': NetworkxDualCatenation,
        'crossMesh': NetworkxDualCrossMesh,
    }

    if dual_type in plexus_mode:
        dual_graph = plexus_mode[dual_type](num_periods)
    else:
        print('Warning: Invalid graph mode, choose default')
        dual_graph = plexus_mode['default'](num_periods)

    return dual_graph

def construct_from_graphSet(graphSet, circuit_type='default'):

    circuitConstructor = {
        'default': Circuit,
        'circuit': Circuit,
        'flow': FlowCircuit,
        'flux': FluxCircuit,
        }

    circuitSet = []
    if circuit_type in circuitConstructor:

        for g in graphSet.layer:
            K = circuitConstructor[circuit_type](g)

            circuitSet.append(K)

    else:
        print('Warning: Invalid graph mode, choose default')

        for g in graphSet.layer:
            K = circuitConstructor['default'](g)
            circuitSet.append(K)

    return circuitSet

def initialize_dual_from_catenation(dual_type='catenation', num_periods=1,
        circuit_type='default'):

    """
    Initialize a dual spatially embedded circuit, with internal graphs based on
    simple catenatednetwork skeletons.

    Args:
        dual_type (string):\n
            The type of dual skeleton (simple, diamond, laves, catenation).
        num_periods (int):\n
            Repetition number of the lattice's unit cell.

    Returns:
        dual_circuit: A dual circuit system.

    """

    graphSet = init_dualCatenation(dual_type, num_periods)
    circuitSet = construct_from_graphSet(graphSet, circuit_type)
    for i, g in enumerate(graphSet.layer):
        circuitSet[i].info = circuitSet[i].set_info(g, dual_type)

    kirchhoff_dual = DualCircuit(circuitSet)

    return kirchhoff_dual

def initialize_dual_from_minsurf(
        dual_type='simple',
        num_periods=2,
        circuit_type='default'
        ):
    """
    Initialize a dual spatially embedded flux circuit, with internal graphs
    based on the network skeletons of triply-periodic minimal surfaces.

    Args:
        dual_type (string):\n
            The type of dual skeleton (simple, diamond, laves, catenation).
        num_periods (int):\n
            Repetition number of the lattice's unit cell.

    Returns:
        dual_circuit: A flow_circuit object.

    """

    graphSet = init_dual_minsurf_graphs(dual_type, num_periods)
    circuitSet = construct_from_graphSet(graphSet, circuit_type)
    for i, g in enumerate(graphSet.layer):
        circuitSet[i].info = circuitSet[i].set_info(g, dual_type)
    kirchhoff_dual = DualCircuit(circuitSet)

    kirchhoff_dual.e_adj = graphSet.e_adj
    kirchhoff_dual.e_adj_idx = graphSet.e_adj_idx
    kirchhoff_dual.distance_edges()

    return kirchhoff_dual

def set_print_labels(D):

    for i, khf in enumerate(D.layer):

        khf.nodes['label'] = [n for n in khf.G.nodes()]
        khf.edges['label'] = [e for i, e in enumerate(khf.G.edges())]

    return D

### Visualization

In [ ]:
import plotly
import plotly.graph_objects as go

In [ ]:
###### Visualization ######
def plot_networkx(input_graph, **kwargs):

    """
    Return an interactive network plot, which shows the internal edge and node
     data via hover and text.

    Args:
        input_graph (nx.Graph):\n
            A networkx graph.
        kwargs (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.

    Returns:
        plotly.graph_objects.Figure: A plotly figure object

    """

    options = {
        'network_id': 0,
        'color_nodes': ['#a845b5'],
        'color_edges': ['#c762d4'],
        'colormap': ['plasma'],
        'markersize': [2],
        'linewidth': [5],
        'axis': True
    }

    node_data = pd.DataFrame()
    edge_data = pd.DataFrame()

    for k, v in kwargs.items():
        if k in options:
            options[k] = v
    if 'node_data' in kwargs:
        for sk, sv in kwargs['node_data'].items():
            node_data[sk] = sv.to_numpy()
    if 'edge_data' in kwargs:
        for sk, sv in kwargs['edge_data'].items():
            edge_data[sk] = sv.to_numpy()

    fig = go.Figure()
    add_traces_nodes(fig, options, input_graph, node_data)
    add_traces_edges(fig, options, input_graph, edge_data)

    update_layout(fig, options)

    return fig


def update_layout(fig, options):

    new_layout = {
        'showlegend': False,
        'autosize': True,
        'margin': {'l': 0, 'r': 0, 't': 0, 'b': 0},
        'scene': dict(aspectmode="data",)
    }
    if not options['axis']:
        new_layout.update(
            dict(
                plot_bgcolor="#FFF",  # Sets background color to white
                xaxis=dict(
                    linecolor="#BCCCDC",  # Sets color of X-axis line
                    showgrid=False  # Removes X-axis grid lines
                ),
                yaxis=dict(
                    linecolor="#BCCCDC",  # Sets color of Y-axis line
                    showgrid=False,  # Removes Y-axis grid lines
                ),
            )
        )

    fig.update_layout(**new_layout)


# integrate traces into the figure
def add_traces_edges(fig, options, input_graph, extra_data):

    """
    Add line traces for interactive edge data display.

    Args:
        fig (plotly.graph_objects.Figure:\n
            A plotly figure object
        options (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.
        input_graph (nx.Graph):\n
            A networkx graph.
        extra_data (pandas.DataFrame):\n
            A dataframe holding the data.

    """

    idx = options['network_id']

    optM = {
        'color': options['color_edges'][idx],
        'colormap': options['colormap'][idx]
    }
    edge_mid_trace = get_edge_mid_trace(input_graph, extra_data, **optM)

    optI = {
        'color': options['color_edges'][idx],
        'linewidth': options['linewidth'][idx],
        'colormap': options['colormap'][idx]
    }

    edge_invd_traces = get_edge_invd_traces(input_graph, extra_data, **optI)

    for eit in edge_invd_traces:
        fig.add_trace(eit)

    fig.add_trace(edge_mid_trace)


def add_traces_nodes(fig,  options,  input_graph, extra_data):

    """
    Add point traces for interactive node data display.

    Args:
        fig (plotly.graph_objects.Figure:\n
            A plotly figure object
        options (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.
        input_graph (nx.Graph):\n
            A networkx graph.
        extra_data (pandas.DataFrame):\n
            A dataframe holding the data.

    """

    idx = options['network_id']
    optN = {
        'color': options['color_nodes'][idx],
        'markersize': options['markersize'][idx],
        'colormap': options['colormap'][idx]
    }

    node_trace = get_node_trace(input_graph,  extra_data, **optN)
    fig.add_trace(node_trace)


In [ ]:
###### Visualization (Dual) ######
def plot_networkx_dual(dual_circuit, *args, **kwargs):
    """
    Return an interactive network plot, which shows the internal edge and node
     data via hover and text for a multilayer system.

    Args:
        dual_circuit (dual_circuit):\n
            A dual_circuit object.
        args (list):\n
            A list for keywoprd data for display.
        kwargs (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.

    Returns:
        plotly.graph_objects.Figure: A plotly figure object

    """

    options = {
        'network_id': 0,
        # 'color_nodes': ['#750615', '#08379c'],
        # 'color_edges': ['#d12139', '#2159d1'], # red and blue
        'color_nodes': ['#3f3f40', '#3f3f40'],
        'color_edges': ['#8a8a8a', '#2159d1'], # grey
        'colormap': ['plasma', 'plasma'],
        'markersize': [8, 8],
        'linewidth': [10, 10],
        'axis': True,
    }

    for k, v in kwargs.items():
        if k in options:
            options[k] = v

    fig = go.Figure()

    for i, K in enumerate(dual_circuit.layer):

        options['network_id'] = i
        node_data = K.get_nodes_data(*args)
        edges_data = K.get_edges_data(*args)

        add_traces_edges(fig, options, K.G, edges_data)
        add_traces_nodes(fig, options, K.G, node_data)


    update_layout_dual(fig, options)

    return fig

def update_layout_dual(fig, options):

    new_layout = {
        'showlegend': False,
        'autosize': True,
        'margin': {'l': 0, 'r': 0, 't': 0, 'b': 0},
        'scene_camera': dict(eye=dict(x=2, y=2, z=0.9)),
        'scene': dict(aspectmode="data"),
    }

    axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=False,
        )

    if not options['axis']:
        new_layout.update(
            {
                'scene': dict(
                    xaxis_title='',
                    yaxis_title='',
                    zaxis_title='',
                    xaxis=axisFormat,
                    yaxis=axisFormat,
                    zaxis=axisFormat,
                    aspectmode="data",
                )
            }
        )

    fig.update_layout(**new_layout)


# integrate traces into the figure
def add_traces_edges(fig, options, input_graph, extra_data):

    """
    Add line traces for interactive edge data display.

    Args:
        fig (plotly.graph_objects.Figure:\n
            A plotly figure object
        options (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.
        input_graph (nx.Graph):\n
            A networkx graph.
        extra_data (pandas.DataFrame):\n
            A dataframe holding the data.

    """

    idx = options['network_id']

    optM = {
        'color': options['color_edges'][idx],
        'colormap': options['colormap'][idx]
    }
    edge_mid_trace = get_edge_mid_trace(input_graph, extra_data, **optM)

    optI = {
        'color': options['color_edges'][idx],
        'linewidth': options['linewidth'][idx],
        'colormap': options['colormap'][idx]
    }

    edge_invd_traces = get_edge_invd_traces(input_graph, extra_data, **optI)

    for eit in edge_invd_traces:
        fig.add_trace(eit)

    fig.add_trace(edge_mid_trace)


def add_traces_nodes(fig,  options,  input_graph, extra_data):

    """
    Add point traces for interactive node data display.

    Args:
        fig (plotly.graph_objects.Figure:\n
            A plotly figure object
        options (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.
        input_graph (nx.Graph):\n
            A networkx graph.
        extra_data (pandas.DataFrame):\n
            A dataframe holding the data.

    """

    idx = options['network_id']
    optN = {
        'color': options['color_nodes'][idx],
        'markersize': options['markersize'][idx],
        'colormap': options['colormap'][idx]
    }

    node_trace = get_node_trace(input_graph,  extra_data, **optN)
    fig.add_trace(node_trace)

# auxillary functions generating traces for nodes and edges
def get_edge_mid_trace(input_graph, extra_data, **kwargs):

    """
    Return transparent line traces for interactive edge data display.

    Args:
        input_graph (nx.Graph):\n
            A networkx graph.
        extra_data (pandas.DataFrame):\n
            A dataframe holding the data.
        kwargs (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.

    Returns:
        plotly.graph_objects.Scatter: A plotly scatter object

    """

    options = {
        'color': '#888',
        # 'dim':3
    }
    dim = 3
    for k, v in kwargs.items():
        if k in options:
            options[k] = v

    pos = nx.get_node_attributes(input_graph, 'pos')
    if len(list(pos.values())[0]) != dim:
        dim = len(list(pos.values())[0])

    E = input_graph.edges()
    # if 'edge_list' in options:
    #     E = options['edge_list']

    middle_node_trace = get_hover_scatter_from_template(dim, options)

    XYZ = [[] for i in range(dim)]
    for j, edge in enumerate(E):

        XYZ_0 = pos[edge[0]]
        XYZ_1 = pos[edge[1]]

        for i, xi in enumerate(XYZ):
            xi.append((XYZ_0[i]+XYZ_1[i])/2.)

    set_hover_info(middle_node_trace, XYZ, extra_data)

    return middle_node_trace

def set_hover_info(trace, XYZ, extra_data):

    """
    Set hover info for figure traces.

    Args:
        trace (plotly.graph_objects.trace): A networkx graph.
        XYZ (ndarray): Nodal position data.
        extra_data (pandas.DataFrame): A dataframe holding the data.

    """

    tags = ['x', 'y', 'z']
    if len(XYZ) < 3:
        tags = ['x', 'y']
    for i, t in enumerate(tags):
        trace[t] = XYZ[i]

    if len(extra_data.keys()) != 0:
        data = [list(extra_data[c]) for c in extra_data.columns]
        iter = list(zip(*data))
        text = [create_tag(vals, extra_data.columns) for vals in iter]
        trace['text'] = text
    else:
        trace['hoverinfo'] = 'none'

    trace['hoverlabel'] = dict(bgcolor="white")


def get_hover_scatter_from_template(dim, options):

    """
    Get scatter hover info for figure traces.

    Args:
        dim (int):\n
            A dimensional identifier.
        options (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.

    Returns:
        plotly.graph_objects.Scatter: A plotly scatter object

    """

    if dim == 3:
        middle_node_trace = go.Scatter3d(
            x=[],
            y=[],
            z=[],
            text=[],
            mode='markers',
            hoverinfo='text',
            opacity=0,
            marker=dict(**options)
            # marker = dict(color = options['color'])
        )
    else:
        middle_node_trace = go.Scatter(
            x=[],
            y=[],
            text=[],
            mode='markers',
            hoverinfo='text',
            marker=go.scatter.Marker(
                opacity=0,
                **options
            )

        )

    return middle_node_trace


def get_edge_invd_traces(input_graph, extra_data,  **kwargs):

    """
    Return individual line traces for interactive edge data display.

    Args:
        input_graph (nx.Graph):\n
            A networkx graph.
        extra_data (pandas.DataFrame):\n
            A dataframe holding the data.
        kwargs (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.

    Returns:
        plotly.graph_objects.Scatter: A plotly scatter object

    """

    options = {
        'color': '#888',
        'width': 2,
        # 'dim':3
    }
    # options['width'] = kwargs['linewidth']
    dim = 3
    for k, v in kwargs.items():
        if k in options:
            options[k] = v

    # handle exceptions and new containers
    colorful = False
    if type(options['color']) != str:
        colorful = True
        cmax = np.max(options['color'])
        # cmin = np.min(options['color'])
        if cmax == 0:
            cmax = 1.

        pc = plotly.colors.sample_colorscale(
                kwargs['colormap'],
                options['color']/cmax
                )
        options['color'] = pc

    weighted = False
    if 'linewidth' in kwargs.keys():
        options['width'] = kwargs['linewidth']
    if type(options['width']) != int:
        weighted = True

    pos = nx.get_node_attributes(input_graph, 'pos')
    if len(list(pos.values())[0]) != dim:
        dim = len(list(pos.values())[0])

    E = input_graph.edges()
    # if 'edge_list' in options:
    #     E = options['edge_list']

    # add new traces
    trace_list = []
    aux_option = dict(options)

    for i, edge in enumerate(E):

        if colorful:
            aux_option['color'] = options['color'][i]

        if weighted:
            aux_option['width'] = options['width'][i]

        trace = get_line_from_template(dim, aux_option)
        XYZ_0 = input_graph.nodes[edge[0]]['pos']
        XYZ_1 = input_graph.nodes[edge[1]]['pos']

        set_edge_info(trace, XYZ_0, XYZ_1)
        trace_list.append(trace)

    return trace_list


def set_edge_info(trace, XYZ_0, XYZ_1):

    """
    Set hover info for figure traces.

    Args:
        trace (plotly.graph_objects.trace): A networkx graph.
        XYZ_0 (ndarray): Nodal position data.
        XYZ_0 (ndarray): Nodal position data.
    """

    tags = ['x', 'y', 'z']
    if len(XYZ_0) < 3:
        tags = ['x', 'y']

    for i, t in enumerate(tags):
        trace[t] = [XYZ_0[i],  XYZ_1[i],  None]


def get_line_from_template(dim, options):

    """
    Get a line trace element.

    Args:
        dim (int):\n
            A dimensional identifier.
        options (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.

    Returns:
        plotly.graph_objects.Scatter: A plotly scatter object

    """

    if dim == 3:

        trace = go.Scatter3d(
            x=[],
            y=[],
            z=[],
            mode='lines',
            line=dict(**options),
            hoverinfo='none'
        )

    else:

        trace = go.Scatter(
            x=[],
            y=[],
            mode='lines',
            line=dict(**options),
            hoverinfo='none'
        )

    return trace


def get_node_trace(input_graph, extra_data, **kwargs):

    """
    Return nodal traces for interactive node data display.

    Args:
        input_graph (nx.Graph):\n
            A networkx graph.
        extra_data (pandas.DataFrame):\n
            A dataframe holding the data.
        kwargs (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.

    Returns:
        plotly.graph_objects.Scatter: A plotly scatter object

    """

    options = {
        'color': '#888',
        'dim': 3,
        'markersize': 2
    }

    for k, v in kwargs.items():
        if k in options:

            options[k] = v

    node_xyz = get_node_coords(input_graph, options)

    node_trace = get_node_scatter(node_xyz, extra_data, options)

    return node_trace


def get_node_coords(input_graph, options):

    """
    Return nodal coordinates in specified tuple format.

    Args:
        input_graph (nx.Graph):\n
            A networkx graph.
        options (dictionary):\n
            A dictionary for circuit keywords for readout.

    Returns:
        list: A list of nodal postions [X, Y, Z]

    """

    pos = nx.get_node_attributes(input_graph, 'pos')
    if len(list(pos.values())[0]) != options['dim']:
        options['dim'] = len(list(pos.values())[0])

    node_xyz = [[] for i in range(options['dim'])]

    N = input_graph.nodes()
    if 'node_list' in options:
        N = options['edge_list']

    for node in N:

        xyz_0 = pos[node]

        for i in range(options['dim']):

            node_xyz[i].append(xyz_0[i])

    return node_xyz


def get_node_scatter(node_xyz, extra_data, options):

    """
    Return nodal traces for interactive node data display.

    Args:
        node_xyz (list):\n
            A list of nodal postions [X, Y, Z].
        extra_data (pandas.DataFrame):\n
            A dataframe holding the data.
        options (dictionary):\n
            A dictionary for plotly keywords customizing the plots' layout.

    Returns:
        plotly.graph_objects.Scatter: A plotly scatter object

    """

    mode = 'none'
    hover = ''

    if len(extra_data.keys()) != 0:
        mode = 'text'
        data = [list(extra_data[c]) for c in extra_data.columns]
        iter = list(zip(*data))
        hover = [create_tag(vals, extra_data.columns) for vals in iter]

    if options['dim'] == 3:
        node_trace = go.Scatter3d(
            x=node_xyz[0],
            y=node_xyz[1],
            z=node_xyz[2],
            mode='markers',
            hoverinfo=mode,
            hovertext=hover,
            marker=dict(
                size=options['markersize'],
                line_width=2,
                color=options['color'])
        )
    else:
        node_trace = go.Scatter(
            x=node_xyz[0],
            y=node_xyz[1],
            mode='markers',
            hoverinfo=mode,
            hovertext=hover,
            marker=dict(
                size=options['markersize'],
                line_width=2,
                color=options['color'])
        )

    return node_trace

def create_tag(vals, columns):

    """
    Create a hover tag.

    Args:
        vals (list): A list of values.
        columns (list):A list of keywords.

    Returns:
        string: A string in makrdown format.

    """

    tag = ''
    for i, c in enumerate(columns):
        tag += str(c)+': '+str(vals[i])+'<br>'

    return tag

In [ ]:
def plot_networkx_custom(G, colors_edges=['#616366', '#d12139']):
    """Plot the networkx Graph, separate positive and negative edges
    """

    # Extract node coordinates
    node_x = []
    node_y = []
    node_z = []
    node_colors = []
    for node in G.nodes():
        x, y, z = G.nodes[node]['pos']
        node_x.append(x)
        node_y.append(y)
        node_z.append(z)
        node_colors.append(G.nodes[node]['color'])

    # Create node trace
    node_trace = go.Scatter3d(
        x=node_x, y=node_y, z=node_z,
        mode='markers+text',
        marker=dict(size=8, color=node_colors),
        text=[str(n) for n in G.nodes()],
        textposition='top center',
        name='Nodes',
        hoverinfo='text',
        showlegend=False
    )

    # Create edge traces (split by sign to get separate legend items)
    # colors_edges = ['#616366', '#d12139'] #'#8a8a8a'
    positive_edges = [(u, v) for u, v in G.edges() if G[u][v]['weight'] > 0]
    negative_edges = [(u, v) for u, v in G.edges() if G[u][v]['weight'] < 0]
    # print(positive_edges)
    # print(negative_edges)

    # Positive edge trace (green, legend on)
    pos_edge_trace = go.Scatter3d(
        x=sum([[G.nodes[u]['pos'][0], G.nodes[v]['pos'][0], None] for u, v in positive_edges], []),
        y=sum([[G.nodes[u]['pos'][1], G.nodes[v]['pos'][1], None] for u, v in positive_edges], []),
        z=sum([[G.nodes[u]['pos'][2], G.nodes[v]['pos'][2], None] for u, v in positive_edges], []),
        mode='lines',
        line=dict(color=colors_edges[0], width=10),
        name='+'
    )

    # Negative edge trace (red, legend on)
    neg_edge_trace = go.Scatter3d(
        x=sum([[G.nodes[u]['pos'][0], G.nodes[v]['pos'][0], None] for u, v in negative_edges], []),
        y=sum([[G.nodes[u]['pos'][1], G.nodes[v]['pos'][1], None] for u, v in negative_edges], []),
        z=sum([[G.nodes[u]['pos'][2], G.nodes[v]['pos'][2], None] for u, v in negative_edges], []),
        mode='lines',
        line=dict(color=colors_edges[1], width=10),
        name='-'
    )

    # Create the final figure
    fig = go.Figure(data=[node_trace, pos_edge_trace, neg_edge_trace])

    # Update layout
    fig.update_layout(
        # title="Interactive 3D Signed Path Graph",
        # scene=dict(
        #     xaxis=dict(range=[0, 3], title='x'),
        #     yaxis=dict(range=[0, 1], title='y'),
        #     zaxis=dict(range=[-0.5, 0.5], title='z'),
        # ),
        margin=dict(l=0, r=0, b=0, t=30),
        showlegend=True,
        legend=dict(
            x=0.62, y=0.6,
            xanchor='center',
            yanchor='middle',
            bgcolor='rgba(255,255,255,0.7)',  # semi-transparent white background
            bordercolor='black',
            borderwidth=1
        )
    )

    # Update layout
    new_layout = {
            'autosize': True,
            'margin': {'l': 0, 'r': 0, 't': 0, 'b': 0},
            'scene_camera': dict(eye=dict(x=2, y=2, z=0.9)),
            'scene': dict(aspectmode="data"),
        }

    axisFormat = dict(
                showbackground=False,
                showticklabels=False,
                autorange=True,
                showgrid=False,
        )


    fig.update_layout(**new_layout)

    # Show interactive plot
    fig.show()

### Ensnarlment

In [ ]:
###### simpleCycles ######
import random as rd


class simple_cycles(object):

    """
    A class to generate cycle bases and create a minimal basis in
    networkx formats

    Attributes
    ----------
    G : nx.Graph()
        A simple graph which on which the cycle basis is calcuclated
    """

    def __init__(self, G=None):

        self.G = G

    def generate_cycle_lists(self):

        """
        Returns an edge list, and labeled dictionary of cycles drawn from
        a Horton cycle search for all vertices.

        Returns:
            dictionary: \n
                A dictionary which of cycles generated from bfs searches
            list: \n
                a list of cycles represented by their edge sets

        Raises
        -------
        NotImplementedError
            If no graph is initially set for the backbone Graph G.
        """

        if self.G is None:
            raise RuntimeError(
                "cycle_tools_simple.simple_cycles.G is not set!"
                )

        nx.set_node_attributes(self.G, False, 'push')

        # check for graph_type, then check for paralles in the Graph,
        # if existent insert dummy nodes to resolve conflict,
        # cast the network onto simple graph afterwards
        for i, e in enumerate(self.G.edges()):
            self.G.edges[e]['label'] = i

        root_sets = []
        for n in self.G.nodes():
            # building new tree using breadth first
            root_sets.append(self.compute_cycles_superlist(n))

        key = 0
        cyc_dict = {}
        cyc_list = {}
        for cyc_sets in root_sets:
            for cyc_E in cyc_sets:
                # relabeling and weighting graph
                cyc_list.update({key: cyc_E})

                labels = [self.G.edges[f]['label'] for f in cyc_E]
                cyc_dict.update({key: labels})

                key += 1

        return cyc_dict, cyc_list

    def find_cycle(self, dict_path, e, n):
        """
        Returns an edge list, and node list for a cycle constructed from
        spanning tree + additional edge.

        Args:
            dict_path (dictionary):\n
                A dictionary of shortest paths in the bfs tree
            e (tuple):\n
                The edge which is to be plugge into the bfs tree and generates
                a cycle.
            n (int):\n
                The root of the current bfs tree
        Returns:
            list:\n
                A list of vertices for the new cycle
            list:\n
                A list of edges for the new cycle
        """

        # label pathways
        l1 = dict_path[e[1]][::-1]
        l2 = dict_path[e[0]][::-1]
        if len(dict_path[e[0]]) < len(dict_path[e[1]]):
            l1 = dict_path[e[0]][::-1]
            l2 = dict_path[e[1]][::-1]

        idx1 = 0
        idx2 = 0
        for i, n in enumerate(l1):
            if n in l2:
                idx1 = i
                idx2 = l2.index(n)
                break
        L2 = l2[:idx2]

        new_path = l1[:idx1+1]+L2[::-1]
        new_edges = [(p, new_path[i+1]) for i, p in enumerate(new_path[:-1])]
        new_edges += [e]

        return new_path, new_edges

    def compute_cycles_superlist(self, root):
        """
        Returns an edge list of cycles drawn from a Horton cycle search for
        one vertex.

        Args:
            root (int): The root vertex of the current bfs tree

        Returns:
            list: The superlist of cycles from all bfs trees, in edge list
            representation

        Raises:
            Exception: description

        """

        spanning_tree, dict_path = self.breadth_first_tree(root)
        diff_graph = nx.difference(self.G, spanning_tree)
        list_cycles = []
        for e in diff_graph.edges():

            simple_cycle, cycle_edges = self.find_cycle(dict_path, e, root)
            list_cycles.append(cycle_edges)

        return list_cycles

    def construct_networkx_minimum_basis(self, input_graph=None):

        """
        Return a minimum cycle basis for the input graph, with all elements
        edge lists.

        Args:
            input_graph (nx.Graph):\n
                A networkx graph with 'many' cycles

        Returns:
            list:\n
                The minimal basis of the graph, represented by a list of
                networkx graphs.
        """
        if input_graph is not None:
            self.G = nx.Graph(input_graph)

        C = self.construct_minimum_basis()

        networkx_basis = self.fillInGraphs(C)

        return networkx_basis

    def construct_networkx_basis(self, input_graph=None, root='random'):

        """
        Return a cycle basis for the input graph, with all elements
        edge lists.

        Args:
            input_graph (nx.Graph):\n
                A networkx graph with 'many' cycles

        Returns:
            list:\n
                The basis of the graph, represented by a list of networkx
                graphs.
        """
        if root == 'random':
            N = list(self.G.nodes)
            rnd = rd.randint(0, len(N)-1)
            root = N[rnd]
        if input_graph is not None:
            self.G = nx.Graph(input_graph)

        C = self.compute_cycles_superlist(root)

        networkx_basis = self.fillInGraphs(C)

        return networkx_basis

    def fillInGraphs(self, C):

        networkx_basis = []
        for cs in C:
            new_cycle = nx.Graph()
            for e in cs:

                new_cycle.add_edge(*e)
                for k, v in self.G.edges[e].items():
                    new_cycle.edges[e][k] = v

            for n in new_cycle.nodes():

                for k, v in self.G.nodes[n].items():
                    new_cycle.nodes[n][k] = v

            networkx_basis.append(new_cycle)

        return networkx_basis

    def construct_minimum_basis(self):

        """
        Return a cycle basis for the input graph, with all elements
        edge lists.

        Args:
            input_graph (nx.Graph):\n
                A networkx graph

        Returns:
            list:\n
                The minimal basis of the graph, represented by a list of edge
                lists.

        Raises:
            Exception: description

        """

        # calc minimum weight basis and construct dictionary for weights of
        # edges, takes a leave-less, connected, N > 1 SimpleGraph as input,
        # no self-loops optimally, deviations are not raising any warnings
        # sort basis vectors according to weight, creating a new minimum weight
        # basis from the total_cycle_list
        P = nx.number_connected_components(self.G)
        nullity = nx.number_of_edges(self.G)-nx.number_of_nodes(self.G)+P

        cyc_dict, cyc_list = self.generate_cycle_lists()
        cyc_len = {}
        for c, e in cyc_dict.items():
            cyc_len[c] = len(e)
        sorted_cycle_list = sorted(cyc_len, key=cyc_len.__getitem__)

        min_basis = []
        min_label = []
        EC = nx.Graph()
        counter = 0

        for c in sorted_cycle_list:

            cycle_edges_in_basis = True
            new_cycle = cyc_list[c]

            for e in new_cycle:
                if not EC.has_edge(*e):
                    EC.add_edge(*e, label=counter)
                    counter += 1
                    cycle_edges_in_basis = False

            # if cycle edges where not part of the supergraph yet then it
            # becomes automatically part of the basis
            if not cycle_edges_in_basis:

                min_basis.append(new_cycle)
                aux_label = [EC.edges[e]['label'] for e in new_cycle]
                min_label.append(aux_label)

            # if cycle edges are already included we check for linear dependece
            else:
                E = self.edge_matrix(EC, min_label, new_cycle)

                linear_independent = self.compute_linear_independence(E)

                if linear_independent:
                    min_basis.append(new_cycle)
                    aux_label = [EC.edges[e]['label'] for e in new_cycle]
                    min_label.append(aux_label)

            if len(min_basis) == nullity:
                break

        if len(min_basis) < nullity:
            raise RuntimeError('Construction error, not enough cycles found!')

        return min_basis

    def edge_matrix(self, nx_edges, minimum_label, new_cycle):
        """
        Return a binary matrix for operations on Z2, representing current
        cycle candidates and a test cycle.

        Args:
            nx_edges (nx.Graph):\n
                A networkx graph backbone being rebuilt with cycle base edges
            minimum_label (list):\n
                The labels sorting the edges in the binary cycle matrix.
            new_cycle (list):\n
                A list of edges of the cycle to be tested.

        Returns:
            ndarray: Numpy array representing a binary cycle matrix in Z2.

        Raises:
            Exception: description

        """

        rows = len(nx_edges.edges())
        length_basis = len(minimum_label)
        columns = length_basis+1
        E = np.zeros((rows, columns))

        for i in range(length_basis):
            E[minimum_label[i], i] = 1

        for m in new_cycle:
            E[nx_edges.edges[m]['label'], -1] = 1

        return E

    def compute_linear_independence(self, edge_mat):

        """
        Return bool whether all columns of E are linear independent in Z2.

        Args:
            edge_mat (ndarray):\n
                An ndarray representing a binary cycle matrix in Z2.

        Returns:
            bool:\n
                Result indicating whether the columns are linear independent.

        Raises:
            Exception: description

        """

        linear_independent = False
        columns = len(edge_mat[0, :])

        # calc echelon form
        a_columns = np.arange(columns-1)
        for col in a_columns:
            idx_nz = np.nonzero(edge_mat[col:, col])[0]
            idx = idx_nz[0]+col

            if len(idx_nz) == 1:

                edge_mat[[col, idx], :] = edge_mat[[idx, col], :]

            else:

                new_idx = idx_nz[1:]+col
                aux_E = np.add(edge_mat[new_idx], edge_mat[idx])
                edge_mat[new_idx] = np.mod(aux_E, 2)
                edge_mat[[col, idx], :] = edge_mat[[idx, col], :]

        r = np.nonzero(edge_mat[columns-1:, -1])[0]
        if r.size:
            linear_independent = True

        return linear_independent

    def breadth_first_tree(self, root):

        """
        Return a bfs-tree from root, as well a dictionary of shortest paths
        between branching points and leaves.

        Args:
            root (int):\n
                The root vertex for bfs search.

        Returns:
            nx.Graph:\n
                The spanning tree from bfs search
            dictionary:\n
                A dicitonary of shortest paths between branching points and
                leaves.

        """

        T = nx.Graph()
        push_down = nx.get_node_attributes(self.G, 'push')
        len_n = len(self.G.nodes())

        if len(push_down.keys()) != len_n:
            push_down = {}
            for n in self.G.nodes():
                push_down[n] = False

        push_down[root] = True
        root_queue = []

        labels = self.G.edges(root)
        dict_path = {root: [root]}

        args = [root, T, labels, push_down, dict_path, root_queue]
        self.compute_sprouts(*args)

        while T.number_of_nodes() < len_n:
            new_queue = []
            for q in root_queue:

                labels = self.G.edges(q)
                args = [q, T, labels, push_down, dict_path, new_queue]
                self.compute_sprouts(*args)

            root_queue = new_queue[:]

        return T, dict_path

    def compute_sprouts(self, root, T, labels, push_down, dict_path, queue):

        """
        Update bfs push list and tree structure.
        """

        for e in labels:

            if e[0] == root:
                if not push_down[e[1]]:
                    T.add_edge(*e)
                    queue.append(e[1])
                    push_down[e[1]] = True
                    dict_path[e[1]] = dict_path[root]+[e[1]]
            else:
                if not push_down[e[0]]:
                    T.add_edge(*e)
                    queue.append(e[0])
                    push_down[e[0]] = True
                    dict_path[e[0]] = dict_path[root]+[e[0]]

    def extract_path_origin(self, cycle):

        """
        Find and return an oriented closed edge walk on a simple cycle.

        Args:
            cycle (nx.Graph):\n
                A networkx graph representing a simple Eulerian cycle.

        Returns:
            list:\n
                A list of nodes, in order of the cyclic path.

        """

        path = []
        ep = nx.eulerian_path(cycle, source=list(cycle.nodes())[0])

        for i, e in enumerate(ep):
            path.append(cycle.nodes[e[0]]['pos'])

        return path

In [ ]:
###### CycleToolsLinking ######
import itertools as itl
from scipy import interpolate
from scipy.integrate import nquad

@dataclass
class linkedCycles_tools(simple_cycles):

    """
    A class f function tools to compte linking of cycle basis of simple,
    spatially embedded graphs.
    Attributes:
        tck (dict):\n
            Spline parameters for curve smoothening
        edge_res (int):\n
            Edge point resolution.
        XS (list):\n
            Array of curve parameters (0,1) for edge point densification.
        N (int):\n
            ???
        res (int):\n
            Numeric resoltuion of Gauss map (double integral) evaluation.
        threshold (float):\n
            Linkage number demanded accuracy.
        x (list):\n
            An array holding curve parameters for smoothened curves.
        dxy (float):\n
            Double integral infinitesimal square factor.

    Methods:
        __post_init__()
            The model post_init function.
        calc_linkage_cycleSets_nxGraph(cycle_sets1, cycle_sets2)
            Compute the linkage dictionaries in boolean and numeric form for
            two sets of cycle bases (in networkx format).
        extract_points_nxGraph(cycle_sets1, cycle_sets2)
            Extract the polygonial representation of each edge of the given
            graph sets.
        calc_linkage_cycleSets_points3D(curves_set1, curves_set2)
            Compute the linkage dictionaries in boolean and numeric form for
            two sets of cycle bases (in polygonial format).
        calc_linkage_points3D(curves_set1, curves_set2)
            Compute the linkage of polygonial curves in 3D via the Gauss map
            for each cycle pairing.
        get_geoInfo(curve_sets)
            Compute cycle centers and maximal point distance from center (For
            pre-sorting spatially distant cycles which cant'possibly be linked)
        compute_link_number(curve1, curve2):
            Compute the specific link for a pair of curves, via Gauss map
            evaluation.
        get_smooth_curve(points3D)
            Smoothing of polygonial curves in order to improve results for
            sharp bending points and fourther function generator dependencies.
        refine_curve_points(points3D)
            Increase point density of the graph'sedges, by inserting extra
            points along the line segments.
        get_smooth_points(t, tck)
            Create a smooth 3D point representation by utilizing previoulsy
            calucalted spline parameters and current curve parameters.
        get_smooth_director(t, tck):
            Create a smooth 3D point representation of the curves tangent by
            utilizing previoulsy calucalted spline parameters and current curve
            parameters.
        gauss_map(points, tangents)
            Gauss map evaluation utilizing the entirety of curve points and
            their tangents (equal to entire double integral kernel evaluation).

    """
    kwargs = dict(init=False, repr=False)

    tck: dict = field(default_factory=dict, **kwargs)
    edge_res: int = field(default=5, **kwargs)
    XS: list = field(default_factory=list, **kwargs)
    # N: int = field(default=3, **kwargs)
    res: int = field(default=200, **kwargs)
    threshold: float = field(default=0.05, **kwargs)

    x: list = field(default_factory=list, **kwargs)
    dxy: float = field(default=0., **kwargs)

    def __post_init__(self):
        # def __init__(self):
        # super(linkedCycles_tools, self).__init__()
        # self.tck = {}
        # self.edge_res = 5
        self.XS = np.linspace(0, 0.95, self.edge_res)
        # self.N = 3
        # self.res = 200
        # self.threshold = 0.05
        # self.limit = 50

        self.x = np.linspace(0, 1, self.res)
        self.dxy = (self.x[1] - self.x[0])**2

    def calc_linkage_cycleSets_nxGraph(self, cycle_sets1, cycle_sets2):
        """
        Compute the linkage dictionaries in boolean and numeric form for two
        sets of cycle bases (in networkx format).

        Args:
            cycle_sets1 (list): \n
                A list of networkx.Graph objects representing the cycle basis
                of graph #1.
            cycle_sets2 (list): \n
                A list of networkx.Graph objects representing the cycle basis
                of graph #2.
        Returns:
            dict: \n
                A dictionary containing the boolean value of linkage for each
                cycle pair of the input bases.
            dict: \n
                A dictionary containing the numeric value of linkage (Gauss
                map) for each cycle pair of the input bases.
        """

        # precalc curve points and tangents for a given set of curves
        cs = self.extract_points_nxGraph(cycle_sets1, cycle_sets2)
        bool_res, res = self.calc_linkage_cycleSets_points3D(cs[0], cs[1])

        return bool_res, res

    def extract_points_nxGraph(self, cycle_sets1, cycle_sets2):
        """
        Extract the polygonial representation of each edge of the given graph
        sets.

        Args:
            cycle_sets1 (list): \n
                A list of networkx.Graph objects representing the cycle basis
                of graph #1.
            cycle_sets2 (list): \n
                A list of networkx.Graph objects representing the cycle basis
                of graph #2.
        Returns:
            list: \n
                A list of polygons for each input basis representing the
                spatially embeded curve for each basis cycle.

        """
        curves_sets = []
        for i, cs in enumerate([cycle_sets1, cycle_sets2]):

            curves_points = []
            for c in cs:
                cycle_points = self.extract_path_origin(c)
                curves_points.append(cycle_points)

            curves_sets.append(curves_points)

        return curves_sets

    def calc_linkage_cycleSets_points3D(self, curves_set1, curves_set2):
        """
        Compute the linkage dictionaries in boolean and numeric form for two
        sets of cycle bases (in polygonial format).

        Args:
            cycle_sets1 (list): \n
                A list of 3D-point sets representing the cycle basis
                of graph #1.
            cycle_sets2 (list): \n
                A list of 3D-point sets representing the cycle basis
                of graph #2.
        Returns:
            dict: \n
                A dictionary containing the boolean value of linkage for each
                cycle pair of the input bases.
            dict: \n
                A dictionary containing the numeric value of linkage (Gauss
                map) for each cycle pair of the input bases.
        """
        # precalc curve points and tangents for a given set of curves
        res = self.calc_linkage_points3D(curves_set1, curves_set2)
        bool_res = {}

        for kc, vc in res.items():

            if np.round(vc, 0) == 0:
                bool_res[kc] = False
            else:
                bool_res[kc] = True

        return bool_res, res

    def calc_linkage_points3D(self, curves_set1, curves_set2):
        """
        Compute the linkage of polygonial curves in 3D via the Gauss map for
        each cycle pairing.

        Args:
            cycle_sets1 (list): \n
                A list of 3D-point sets representing the cycle basis
                of graph #1.
            cycle_sets2 (list): \n
                A list of 3D-point sets representing the cycle basis
                of graph #2.
        Returns:
            dict: \n
                A dictionary containing the numeric value of linkage (Gauss
                map) for each cycle pair of the input bases.

        """
        res = {}
        curves_sets = [curves_set1, curves_set2]

        # pre-sort non-overlapping cycles
        k = [len(c) for c in curves_sets]
        all_idx = itl.product(range(k[0]), range(k[1]))
        center, dist_fromCenter = self.get_geoInfo(curves_sets)

        check_list = []
        for i, j in all_idx:

            res[(i, j)] = 0.

            dc = np.subtract(center[0][i], center[1][j])
            ds = dist_fromCenter[0][i] + dist_fromCenter[1][j]
            dist_c = np.linalg.norm(dc)
            dist = dist_c-ds

            if dist < 0:
                check_list.append((i, j))

        # compute linking numbers only for spatially overlapping curves
        curve_pairs = [[curves_set1[i], curves_set2[j]] for i, j in check_list]
        # linkage_pairs = itl.starmap(self.compute_link_number, curve_pairs)

        # parallel computing of loop combinations neccessary?
        import multiprocessing as mp
        pool = mp.Pool(processes=4)
        linkage_pairs = pool.starmap(self.compute_link_number, curve_pairs)
        pool.close()
        #
        for i, lk in enumerate(linkage_pairs):
            res.update({check_list[i]: lk})

        return res

    def get_geoInfo(self, curve_sets):

        """
        Compute cycle centers and maximal point distance from center (For
        pre-sorting spatially distant cycles which cant'possibly be linked)

        Args:
            curve_sets (list): \n
                A list of polygons for each input basis representing the
                spatially embeded curve for each basis cycle.

        Returns:
            dict: \n
                A dictionary containing cycle centers for each cycle basis
                (3D points)
            dict: \n
                A dictionary containing maximal point distance from the center
                for each cycle basis.

        """
        center = {0: [], 1: []}
        dist_fromCenter = {0: [], 1: []}

        for i, curves in enumerate(curve_sets):
            for j, cs in enumerate(curves):

                center[i].append(np.mean(cs, axis=0))

                ds = np.subtract(center[i][-1], cs)
                max_ds = np.amax(np.linalg.norm(ds, axis=1))
                dist_fromCenter[i].append(max_ds)

        return center, dist_fromCenter

    def compute_link_number(self, curve1, curve2):
        """
        Compute the specific link for a pair of curves, via Gauss map
        evaluation.

        Args:
            curve_1 (list): \n
                A list 3D points (ordered) forming a closed curve (cycle #1).
            curve_1 (list): \n
                A list 3D points (ordered) forming a closed curve (cycle #2).

        Returns:
            float: \n
                Non-rounded result of numeric double integral evaluation.

        """
        # init path integrals parameters
        xSet = self.x
        dxy = self.dxy
        steps = self.res

        refining = True
        while refining:

            # generate smooth curves for each cycle
            points, tangents = [], []
            for i, curve in enumerate([curve1, curve2]):

                tck = self.get_smooth_curve(curve)
                p = [self.get_smooth_points(xs, tck) for xs in xSet]
                t = [self.get_smooth_director(xs, tck) for xs in xSet]

                points.append(p)
                tangents.append(t)

            lk = self.gauss_map(points, tangents)*dxy/(4.*np.pi)

            # check accuracy, refine path integrals if neccessary
            dl = np.absolute(np.round(lk, 0)-lk)
            if dl < self.threshold:
                refining = False
                break
            else:
                steps *= 2
                xSet = np.linspace(0, 1, steps)
                dx = xSet[1]-xSet[0]
                dxy = dx**2

                if steps > 2000:
                    print('slow convergence, error: ')
                    print(dl)
                    print('refining path integral: ')
                    print(steps)

        return lk

    def get_smooth_curve(self, points3D):
        """
        Smoothing of polygonial curves in order to improve results for sharp
        bending points and fourther function generator dependencies.

        Args:
            points3D (list): \n
                A list 3D points (ordered) forming a closed curve.

        Returns:
            tuple: \n
                (t,c,k) a tuple containing the vector of knots, the B-spline
                coefficients, and the degree of the spline.

        """
        # generate more comprehensive point sets from line curves

        curve_points = self.refine_curve_points(points3D)
        f = np.array(curve_points)

        x = f[:, 0]
        y = f[:, 1]
        z = f[:, 2]

        x[-1] = x[0]
        y[-1] = y[0]
        z[-1] = z[0]

        # smooth curves
        tck, u = interpolate.splprep([x, y, z], s=0., per=1)

        return tck

    def refine_curve_points(self, points3D):
        """
        Increase point density of the graph'sedges, by inserting extra points
        along the line segments.

        Args:
            points3D (list): \n
                A list 3D points (ordered) forming a closed curve, each
                consecutive tuple represents an edge from the graph.

        Returns:
            list: \n
                Denser 3D point representation of closed curves.

        """
        new_curve_points = []
        for i, p1 in enumerate(points3D):

            p2 = points3D[(i+1) % len(points3D)]
            dp = np.subtract(p2, p1)

            new_set = [np.add(x*dp, p1) for x in self.XS]
            new_curve_points += new_set

        return new_curve_points

    def get_smooth_points(self, t, tck):
        """
        Create a smooth 3D point representation by utilizing previoulsy
        calucalted spline parameters and current curve parameters.

        Args:
            t (list): \n
                Current curve parameter.
            tck (tuple): \n
                (t,c,k) a tuple containing the vector of knots, the B-spline
                coefficients, and the degree of the spline.

        Returns:
            ndarrray: \n
                A 3D point as part of the smooth curve representation.

        """
        x, y, z = interpolate.splev(t, tck)

        return np.array((x, y, z))

    def get_smooth_director(self, t, tck):
        """
        Create a smooth 3D point representation of the curves tangent by
        utilizing previoulsy calucalted spline parameters and current curve
        parameters.

        Args:
            t (list): \n
                Current curve parameter.
            tck (tuple): \n
                (t,c,k) a tuple containing the vector of knots, the B-spline
                coefficients, and the degree of the spline.

        Returns:
            ndarrray: \n
                A 3D point as part of the smooth curve representation.

        """
        dt = 0.00001
        p_f = np.array(self.get_smooth_points(t+dt, tck))
        p_b = np.array(self.get_smooth_points(t-dt, tck))

        d = np.subtract(p_f, p_b)/(2. * dt)

        return d

    def gauss_map(self, points, tangents):
        """
        Gauss map evaluation utilizing the entirety of curve points and their
        tangents (equal to entire double integral kernel evaluation).

        Args:
            points (list): \n
                A list of 3D points as part of the smooth curve representation,
                for each cyce.
            tangents (tuple): \n
                A list 3D points as part of the smooth curve representation,
                for each cycle.

        Returns:
            float: \n
                The evaluated, non-rounded value of the Gauss map kernel.

        """
        gm = 0.
        d1 = tangents[0]
        d2 = tangents[1]

        for i, r1 in enumerate(points[0]):

            df = np.subtract(r1, points[1])
            df12 = np.divide(df.transpose(), np.linalg.norm(df, axis=1)**3)
            d12 = np.cross(d1[i], d2)
            gm += np.einsum('ij,ij', d12, df12.transpose())

        return gm

@dataclass
class linkedCycles_extraTools(linkedCycles_tools):
    """
    A class f function tools to compte linking of cycle basis of simple,
    spatially embedded graphs.
    Attributes:
        tck (dict):\n
            Spline parameters for curve smoothening
        edge_res (int):\n
            Edge point resolution.
        XS (list):\n
            Array of curve parameters (0,1) for edge point densification.
        N (int):\n
            ???
        res (int):\n
            Numeric resoltuion of Gauss map (double integral) evaluation.
        threshold (float):\n
            Linkage number demanded accuracy.
        limit (int):\n
            Internal limit numbers.
        lm (list):\n
            List of internal limit numbers.
        itvl (list):\n
            Double integral borders.
        x (list):\n
            An array holding curve parameters for smoothened curves.
        dxy (float):\n
            Double integral infinitesimal square factor.

    Methods:
        __post_init__()
            The model post_init function.
        calc_linkage_cycleSets_nxGraph(cycle_sets1, cycle_sets2)
            Compute the linkage dictionaries in boolean and numeric form for
            two sets of cycle bases (in networkx format).
        extract_points_nxGraph(cycle_sets1, cycle_sets2)
            Extract the polygonial representation of each edge of the given
            graph sets.
        calc_linkage_cycleSets_points3D(curves_set1, curves_set2)
            Compute the linkage dictionaries in boolean and numeric form for
            two sets of cycle bases (in polygonial format).
        calc_linkage_points3D(curves_set1, curves_set2)
            Compute the linkage of polygonial curves in 3D via the Gauss map
            for each cycle pairing.
        get_geoInfo(curve_sets)
            Compute cycle centers and maximal point distance from center (For
            pre-sorting spatially distant cycles which cant'possibly be linked)
        compute_link_number(curve1, curve2):
            Compute the specific link for a pair of curves, via Gauss map
            evaluation.
        get_smooth_curve(points3D)
            Smoothing of polygonial curves in order to improve results for
            sharp bending points and fourther function generator dependencies.
        refine_curve_points(points3D)
            Increase point density of the graph'sedges, by inserting extra
            points along the line segments.
        get_smooth_points(t, tck)
            Create a smooth 3D point representation by utilizing previoulsy
            calucalted spline parameters and current curve parameters.
        get_smooth_director(t, tck):
            Create a smooth 3D point representation of the curves tangent by
            utilizing previoulsy calucalted spline parameters and current curve
            parameters.
        gauss_map(points, tangents)
            Gauss map evaluation utilizing the entirety of curve points and
            their tangents (equal to entire double integral kernel evaluation).
        compute_link_number(curve1, curve2)
            Compute the linking number of two closed curves
            (3D point representation).

    """
    # def __init__(self):
    #
    #     super(linkedCycles_extraTools, self).__init__()
    kwargs = dict(init=False, repr=False)
    limit: int = field(default=50, **kwargs)
    lm: list = field(default_factory=list, **kwargs)
    itvl: list = field(default_factory=list, **kwargs)

    def __post_init__(self):

        self.lm = [dict(limit=self.limit) for i in range(2)]
        self.itvl = [(0, 1), (0, 1)]

    def compute_link_number(self, curve1, curve2):

        """
        Compute the linking number of two closed curves
        (3D point representation).

        Args:
            curve_1 (list): \n
                A list of 3D-point sets representing a cycle
                of graph #1.
            curve_2 (list): \n
                A list of 3D-point sets representing a cycle
                of graph #2.
        Returns:
            dict: \n
                A dictionary containing the numeric value of the linking number
                integral (Gauss map).

        """
        # decompose any polygon train as a series of straigh line functions
        T = []
        for curve in [curve1, curve2]:

            cs = curve+[curve[0]]
            tangents = [
                np.subtract(cs[i+1], ps) for i, ps in enumerate(cs[:-1])
                ]
            T.append(tangents)

        p = itl.product(curve1, curve2)
        t = itl.product(*T)
        pars = zip(p, t)

        # calculate the linking number as evaluation of all line segments
        # combined
        lk = 0.
        for p, t in pars:

            t1, t2 = t
            t12 = np.cross(*t)
            p12 = p[0]-p[1]

            lk += nquad(
                    self.gauss_map,
                    ranges=self.itvl,
                    args=(p12, t, t12),
                    opts=self.lm
                )[0]

        res = lk/(4.*np.pi)

        return res

    def gauss_map(self, x1, x2, p12, t, t12):

        """
        Gauss map piecewise linear bits of the entire closed super-curves.

        Args:
            x1 (float):
                Curve parameter #1.
            x2 (float):
                Curve parameter #2.
            p12 (ndarray):
                Anchor point difference.
            t (list):
                A list of ndarrays representing the current edge tangent
                vector.
            p12 (ndarray):
                Anchor point difference.

        Returns:
            float: \n
                The evaluated, non-rounded value of the Gauss map kernel bit.
        """
        # df = p12+x1*t[0]-x2*t[1]
        # df12 = df/((df[0]**2+df[1]**2+df[2]**2)**1.5)
        # gm = df12[0] * t12[0] + df12[1] * t12[1] + df12[2] * t12[2]

        dx = p12[0]+x1*t[0][0]-x2*t[1][0]
        dy = p12[1]+x1*t[0][1]-x2*t[1][1]
        dz = p12[2]+x1*t[0][2]-x2*t[1][2]

        dr = (dx**2+dy**2+dz**2)**1.5
        gm = (dx * t12[0] + dy * t12[1] + dz * t12[2]) / dr

        return gm

In [ ]:
###### Signature ######

def extract_eulerpath(nx_cycle, root):
    """
    Extract the Eulerian path as an edge sequence from a cycle, given an
    arbitrary starting node.

    Args:
        nx_cycle (networkx.Graph): \n
            A networkx.Graph object represeting a cycle.
        nroot (networkx.Graph.node): \n
            A networkx.Graph.node object to start the path search.
    Returns:
        iterable: \n
            A sequence of edges (ordered)/ a walk along the cyle.
    """

    ep = nx.eulerian_path(nx_cycle, source=root)

    return ep

def get_signature(nx_cycle, sig_graph):
    """
    Given a simple graph for reference, compute the edge signatures for the
    Eulerian path through the cycle.

    Args:
        nx_cycle (networkx.Graph): \n
            A networkx.Graph object represeting a cycle.
        sig_grapht (dict): \n
            A dictionary holding the signature information for any edge in the
            reference graph.
    Returns:
        dict: \n
            A dictionary holding the signature information for any edge in the
            cycle graph.
    """

    sig_cycle = {}

    list_e = list(extract_eulerpath(nx_cycle, list(nx_cycle.nodes())[0]))

    for e in list_e:
        sig_cycle[e] = get_relative_direction(e, sig_graph)

    return sig_cycle

def get_edge_direction(edge_set):
    """
    Setting the intrinsic direction for all edges in a given set.

    Args:
        edge-set (list): \n
            A set of edges, with aplha and omega nodes non-equal.
    Returns:
        dict: \n
            A dictionary holding the signature information for any edge in the
            set.
    """
    sig = {}
    for j, e in enumerate(edge_set):

        sig[e] = [e[0], e[1]]

    return sig

def get_relative_direction(edge, sig_graph):
    """
    Getting the edge signature for an edge set in relation to a reference set.

    Args:
        edge-set (list): \n
            A set of edges, with aplha and omega nodes non-equal.
        sig_grapht (dict): \n
            A dictionary holding the signature information for any edge in the
            reference graph.
    Returns:
        dict: \n
            A dictionary holding the signature information for any edge in the
            set.
    """
    signature = 0
    if edge in sig_graph.keys():
        signature = 1
    else:
        signature = -1

    return signature

In [ ]:
###### edge algebra ######
def edge_column(c, edge_set, sig_graph):
    """
    Compute the signature-sensitive edge vector representation of a given
    cycle.

    Args:
        c (networkx.Graph): \n
            A networkx.Graph representing a Eulerian cycle in the simple
            graph.
        edge_set (list): \n
            A list of all edges in the simple graph.
        sig_graph (dict): \n
            A dictionary holding the information on the graph's intrinsic
            edge signatures (directions) for comparison with cycle edge
            directions.

    Returns:
        nadarray: \n
            The signature-sensitive edge vector representation of the given
            cycle.

    """

    e_row = np.zeros(len(edge_set))
    signa = get_signature(c, sig_graph)

    for i, e in enumerate(edge_set):

        if c.has_edge(*e):

            if e in signa.keys():
                e_row[i] = signa[e]
            else:
                e_row[i] = signa[e[::-1]]

    return e_row


def generate_edge_matrix(basis, edge_set, sig_graph):
    """
    Compute the signature-sensitive edge vector representations of an entire
    cycle basis.

    Args:
        basis (list): \n
            A list networkx.Graph objects, representing the Eulerian cycle
            basis of the simple graph.
        edge_set (list): \n
            A list of all edges in the simple graph.
        sig_graph (dict): \n
            A dictionary holding the information on the graph's intrinsic
            edge signatures (directions) for comparison with cycle edge
            directions.

    Returns:
        nadarray: \n
            The signature-sensitive edge vectors of all cycles bundled into one
            matrix (technically the graph's mesh matrix)
    """
    rows = len(basis)
    cols = len(edge_set)
    E = np.zeros((rows, cols))

    for i, c in enumerate(basis):
        E[i] = edge_column(c, edge_set, sig_graph)

    return E.T

def edge_column_binary(c, edge_set):
    """
    Compute the binary edge vector representation of a given
    cycle.

    Args:
        c (networkx.Graph): \n
            A networkx.Graph representing a Eulerian cycle in the simple
            Graph.
        edge_set (list): \n
            A list of all edges in the simple graph.

    Returns:
        nadarray: \n
            The binary edge vector representation of the given
            cycle.

    """
    e_row = np.zeros(len(edge_set))

    idx = [i for i, e in enumerate(edge_set) if c.has_edge(*e)]
    e_row[idx] = 1

    return e_row

def generate_edge_matrix_binary(basis, edge_set):
    """
    Compute the binary edge vector representations of an entire
    cycle basis.

    Args:
        basis (list): \n
            A list networkx.Graph objects, representing the Eulerian cycle
            basis of the simple graph.
        edge_set (list): \n
            A list of all edges in the simple graph.

    Returns:
        nadarray: \n
            The binary edge vectors of all cycles bundled into one
            matrix (technically the graph's mesh matrix).
    """
    rows = len(basis)
    cols = len(edge_set)
    E = np.zeros((rows, cols))

    for i, c in enumerate(basis):
        E[i] = edge_column_binary(c, edge_set)

    return E

In [ ]:
###### Sample ######

def calc_basisIntertwinedness(graph_sets):
    """
    Compute the first intertwinedness metrics (linkage matrices for arbitrary
    cycle bases).

    Args:
        grap_sets (list):\n
            A list of networkx.Graph objects which posses a defined spatial
            embedding (nodal pos-attributes).

    Returns:
        list:\n
            A list of the arbitrarily chosen cycle matrix for the graphs.
        ndarray:\n
            The linkage matrix of the graph set.

    """

    # generate basis vectors and corresponding graph matrices
    # cyc_nx_bases=[calc_cycle_basis(G) for G in graph_sets]
    cyc_nx_bases = [calc_cycle_minimum_basis(G) for G in graph_sets]

    # compute linkage of basis vectors
    numeric_res = calc_basis_linkage(*cyc_nx_bases)
    lk_mat = extract_linkage_matrix(numeric_res)

    return cyc_nx_bases, lk_mat

def get_basis_matrices(input_graphs, cyc_nx_base):
    """
    Compute auxillary matrix sets for further computation intertwinedness
    metrics.

    Args:
        input_graphs (list):\n
            A list of networkx.Graph objects which posses a defined spatial
            embedding (nodal pos-attributes).
        cyc_nx_base (list):\n
            A list of basis cycles for each graph.

    Returns:
        list:\n
            A list of edge-tuples for each graph.
        list:\n
            The respective edge signatures for each graph.
        list:\n
            A list of (directed) mesh matrices for each graph.
        list:\n
            A list of (undirected) mesh matrices for each graph.

    """
    # get edge representations of graphs
    edge_mat = [list(G.edges()) for G in input_graphs]

    sig = [get_edge_direction(e) for e in edge_mat]

    # init cycle matrices directed and undirected, note they are transposed to
    # each other
    set_size = len(input_graphs)

    cyc_mat_dir = [
        generate_edge_matrix(
            cyc_nx_base[i], edge_mat[i], sig[i]) for i in range(set_size)
        ]

    cyc_mat_undir = [
        generate_edge_matrix_binary(
            cyc_nx_base[i], edge_mat[i]) for i in range(set_size)
        ]

    return edge_mat, sig, cyc_mat_dir, cyc_mat_undir


def calc_cycle_minimum_basis(nx_graph):
    """
    Compute a minimal topological cycle basis of a simple graph via Horton's
    algorithm.

    Args:
        nx_graph (networkx.Graph):\n
            A simple graph.

    Returns:
        list:\n
            A list of networkx.Graph objects, represting the basis cycles of
            the input graph.

    """
    S = linkedCycles_tools()
    S.G = nx.Graph(nx_graph)
    basis_nx = S.construct_networkx_minimum_basis()

    return basis_nx

def extract_linkage_matrix(numeric_res):
    """
    Compute the linkage matrix from raw, non-rounded resuts dict structure.

    Args:
        numeric_res (networkx.Graph):\n
            The linkage dictionary, contaitning the non-rounded results of the
            Gauss-Map calculations for each cycle pair.

    Returns:
        ndarray:\n
            The linkage matrix of two graphs in cycle per cycle representation.

    """
    d1 = set([k[0] for k in numeric_res.keys()])
    d2 = set([k[1] for k in numeric_res.keys()])
    nr = np.zeros((len(d1), len(d2)))

    for i in sorted(d1):
        for j in sorted(d2):
            nr[i, j] = np.round(numeric_res[(i, j)], 0)

    return nr

def calc_basis_linkage(cycle_sets1, cycle_sets2):
    """
    Compute the linkage dictionary for each cycle pair.

    Args:
        cycle_sets1 (list):\n
            A list of networkx.Graph objects, represting the basis cycles of
            the input graph #1.
        cycle_sets2 (list):\n
            A list of networkx.Graph objects, represting the basis cycles of
            the input graph #2.

    Returns:
        dict:\n
            The linkage dictionary, contaitning the non-rounded results of the
            Gauss-Map calculations for each cycle pair.

    """
    # calc linkage of basis cycles
    T = linkedCycles_extraTools()
    bool_res, res = T.calc_linkage_cycleSets_nxGraph(cycle_sets1, cycle_sets2)

    return res

In [ ]:
###### Edge priority ######
import scipy

def getEdgeLinkageOperator(graph_sets):
    """
    Compute the linkage matrices, both in edge space and cycle space
    representation.

    Args:
        graph_sets (list):\n
            A list of networkx.Graph objects.

    Returns:
        ndarray:\n
            The linkage matrix in edge space representation.
        ndarray:\n
            The linkage matrix in cycle space representation.

    """
    cyc_nx_base, lk_mat = calc_basisIntertwinedness(graph_sets)
    graph_matrices = get_basis_matrices(graph_sets, cyc_nx_base)

    cyc_mat_dir = [cm for cm in graph_matrices[-2]]
    cyc_mat_inv = [scipy.linalg.pinv(cm) for cm in cyc_mat_dir]

    P = np.dot(np.dot(cyc_mat_inv[0].T, lk_mat), cyc_mat_inv[1])

    return P, lk_mat

## Main

### Ladder

In [ ]:
# Create spatial networks
num_periods = 3
nx_type = 'catenation'
D = initialize_dual_from_catenation(dual_type=nx_type, num_periods=num_periods)
# D.layer = list of Circuit
## D.layer[0].G = networkx graph

D = set_print_labels(D)

plot_networkx_dual(D)

In [ ]:
# Find cycles
graph_sets = [khf.G for khf in D.layer] # prepare the graph pair

# generate basis vectors
# cyc_nx_bases=[calc_cycle_basis(G) for G in graph_sets]
cyc_nx_bases = [calc_cycle_minimum_basis(G) for G in graph_sets]

In [ ]:
# create graph for cycles: nodes in two types
colors = ['#8a8a8a', '#2159d1'] #'#000308', '#025ff5'
N_cyc1 = len(cyc_nx_bases[0])
N_cyc2 = len(cyc_nx_bases[1])
G_cyc = nx.Graph()

# add nodes for cycles in G1, type 1, position as the average of nodes
for ci in range(N_cyc1):
    pos = np.mean([graph_sets[0].nodes[i]['pos'] for i in cyc_nx_bases[0][ci].nodes()], axis=0)
    G_cyc.add_node(ci, pos=pos, type=1, color=colors[0])
# add nodes for cycles in G2, type 2, position as the average of nodes
for ci in range(N_cyc2):
    pos = np.mean([graph_sets[1].nodes[i]['pos'] for i in cyc_nx_bases[1][ci].nodes()], axis=0)
    G_cyc.add_node(ci+N_cyc1, pos=pos, type=2, color=colors[1])

In [ ]:
# Compute linking number
# compute linkage of basis vectors
numeric_res = calc_basis_linkage(*cyc_nx_bases)
lk_mat = extract_linkage_matrix(numeric_res)

In [ ]:
# Construct the network of cycles
for ci in range(N_cyc1):
    for cj in range(N_cyc2):
        if abs(lk_mat[ci, cj]) > 0:
            G_cyc.add_edge(ci, N_cyc1+cj, weight=lk_mat[ci, cj])

In [ ]:
# Visualize the network of cycles
plot_networkx_custom(G_cyc)

It is clear that for such ladder graphs, the graph of cycles will have such alternating patterns.

In [ ]:
# Construct the network of edges
graph_matrices = get_basis_matrices(graph_sets, cyc_nx_bases)

cyc_mat_dir = [cm for cm in graph_matrices[-2]]
cyc_mat_inv = [scipy.linalg.pinv(cm) for cm in cyc_mat_dir]

P = np.dot(np.dot(cyc_mat_inv[0].T, lk_mat), cyc_mat_inv[1])

In [ ]:
# Visualize the network of edges

In [ ]:
# Visualize the network of nodes

### Diamonds

In [ ]:
num_periods = 5
nx_type = 'diamond'

D = initialize_dual_from_minsurf(dual_type=nx_type, num_periods=num_periods)

D = set_print_labels(D)

for khf in D.layer:
    print("#nodes:", khf.G.number_of_nodes(), "#edges:", khf.G.number_of_edges())

plot_networkx_dual(D)

#nodes: 274 #edges: 430
#nodes: 274 #edges: 430


In [ ]:
# Find cycles
graph_sets = [khf.G for khf in D.layer] # prepare the graph pair

# generate basis vectors
# cyc_nx_bases=[calc_cycle_basis(G) for G in graph_sets]
cyc_nx_bases = [calc_cycle_minimum_basis(G) for G in graph_sets]

In [ ]:
# create graph for cycles: nodes in two types
colors = ['#8a8a8a', '#2159d1'] #'#000308', '#025ff5'
N_cyc1 = len(cyc_nx_bases[0])
N_cyc2 = len(cyc_nx_bases[1])
G_cyc = nx.Graph()

# add nodes for cycles in G1, type 1, position as the average of nodes
for ci in range(N_cyc1):
    pos = np.mean([graph_sets[0].nodes[i]['pos'] for i in cyc_nx_bases[0][ci].nodes()], axis=0)
    G_cyc.add_node(ci, pos=pos, type=1, color=colors[0])
# add nodes for cycles in G2, type 2, position as the average of nodes
for ci in range(N_cyc2):
    pos = np.mean([graph_sets[1].nodes[i]['pos'] for i in cyc_nx_bases[1][ci].nodes()], axis=0)
    G_cyc.add_node(ci+N_cyc1, pos=pos, type=2, color=colors[1])

In [ ]:
# Compute linking number
# compute linkage of basis vectors
numeric_res = calc_basis_linkage(*cyc_nx_bases)
lk_mat = extract_linkage_matrix(numeric_res)

In [ ]:
# Construct the network of cycles
for ci in range(N_cyc1):
    for cj in range(N_cyc2):
        if abs(lk_mat[ci, cj]) > 0:
            G_cyc.add_edge(ci, N_cyc1+cj, weight=lk_mat[ci, cj])

In [ ]:
# Visualize the network of cycles
plot_networkx_custom(G_cyc)

In [ ]:
# Check for cycles - TODO: add labels of nodes + find the corresponding cycles
try:
    cycles = nx.cycle_basis(G_cyc)
    print("Cycle found!")

    for c in cycles:
        sign = 1
        for i in range(len(c)-1):
            sign *= G_cyc.edges[(c[i], c[i+1])]['weight']
        sign *= G_cyc.edges[(c[-1], c[0])]['weight']
        print(sign)
except nx.exception.NetworkXNoCycle:
    print("No cycles found.")

Cycle found!
1.0
1.0
1.0
-1.0
1.0
1.0
-1.0
-1.0
-1.0
-1.0
-1.0
1.0
-1.0
1.0
-1.0
-1.0
1.0
1.0
-1.0
1.0
1.0
-1.0
1.0
-1.0
1.0
-1.0
1.0
1.0
1.0
-1.0
1.0
1.0
1.0
-1.0
1.0
-1.0
-1.0
1.0
-1.0
1.0
-1.0
1.0
1.0
1.0
1.0
-1.0
1.0
1.0
-1.0
1.0
-1.0


Negative cycles occur!

Todos:
- To explore how the number of negative cycles increases as the dimension increases (or the network grows).

### Laves

In [ ]:
num_periods = 2
nx_type = 'laves'
D = initialize_dual_from_minsurf(dual_type=nx_type, num_periods=num_periods)


D = set_print_labels(D)

for khf in D.layer:
    print("#nodes:", khf.G.number_of_nodes(), "#edges:", khf.G.number_of_edges())

plot_networkx_dual(D)

#nodes: 58 #edges: 67
#nodes: 60 #edges: 68


In [ ]:
# Find cycles
graph_sets = [khf.G for khf in D.layer] # prepare the graph pair

# generate basis vectors
# cyc_nx_bases=[calc_cycle_basis(G) for G in graph_sets]
cyc_nx_bases = [calc_cycle_minimum_basis(G) for G in graph_sets]

In [ ]:
# create graph for cycles: nodes in two types
colors = ['#8a8a8a', '#2159d1'] #'#000308', '#025ff5'
N_cyc1 = len(cyc_nx_bases[0])
N_cyc2 = len(cyc_nx_bases[1])
G_cyc = nx.Graph()

# add nodes for cycles in G1, type 1, position as the average of nodes
for ci in range(N_cyc1):
    pos = np.mean([graph_sets[0].nodes[i]['pos'] for i in cyc_nx_bases[0][ci].nodes()], axis=0)
    G_cyc.add_node(ci, pos=pos, type=1, color=colors[0])
# add nodes for cycles in G2, type 2, position as the average of nodes
for ci in range(N_cyc2):
    pos = np.mean([graph_sets[1].nodes[i]['pos'] for i in cyc_nx_bases[1][ci].nodes()], axis=0)
    G_cyc.add_node(ci+N_cyc1, pos=pos, type=2, color=colors[1])

In [ ]:
# Compute linking number
# compute linkage of basis vectors
numeric_res = calc_basis_linkage(*cyc_nx_bases) # these numbers are close to integers (numerical errors)
lk_mat = extract_linkage_matrix(numeric_res)

In [ ]:
# Construct the network of cycles
for ci in range(N_cyc1):
    for cj in range(N_cyc2):
        if abs(lk_mat[ci, cj]) > 0:
            G_cyc.add_edge(ci, N_cyc1+cj, weight=lk_mat[ci, cj])

In [ ]:
# Visualize the network of cycles
plot_networkx_custom(G_cyc)

# Check for cycles - TODO: add labels of nodes + find the corresponding cycles
try:
    cycles = nx.cycle_basis(G_cyc)
    print("Cycle found!")

    for c in cycles:
        sign = 1
        for i in range(len(c)-1):
            sign *= G_cyc.edges[(c[i], c[i+1])]['weight']
        sign *= G_cyc.edges[(c[-1], c[0])]['weight']
        print(c, sign)
except nx.exception.NetworkXNoCycle:
    print("No cycles found.")

Cycle found!
[1, 14, 3, 18] 1.0
[2, 10, 7, 14] 1.0
[5, 10, 7, 14] 1.0
[0, 16, 8, 10, 7, 14] -1.0
[1, 16, 8, 10, 7, 14] -1.0
[0, 11, 8, 10, 7, 14] -1.0
[0, 12, 5, 14] 1.0
[2, 12, 5, 14] 1.0


### Grids

In [ ]:
num_periods = 4
nx_type = 'simple'
D1 = initialize_dual_from_minsurf(dual_type=nx_type, num_periods=num_periods)

d = NetworkxDual()
d.layer = [khf.G for khf in D1.layer]

D = DualCircuit(construct_from_graphSet(d))
D = set_print_labels(D)

for khf in D.layer:
    print("#nodes:", khf.G.number_of_nodes(), "#edges:", khf.G.number_of_edges())

plot_networkx_dual(D)

#nodes: 81 #edges: 108
#nodes: 64 #edges: 144


In [ ]:
# Find cycles
graph_sets = [khf.G for khf in D.layer] # prepare the graph pair

# generate basis vectors
# cyc_nx_bases=[calc_cycle_basis(G) for G in graph_sets]
cyc_nx_bases = [calc_cycle_minimum_basis(G) for G in graph_sets]

In [ ]:
# create graph for cycles: nodes in two types
colors = ['#8a8a8a', '#2159d1'] #'#000308', '#025ff5'
N_cyc1 = len(cyc_nx_bases[0])
N_cyc2 = len(cyc_nx_bases[1])
G_cyc = nx.Graph()

# add nodes for cycles in G1, type 1, position as the average of nodes
for ci in range(N_cyc1):
    pos = np.mean([graph_sets[0].nodes[i]['pos'] for i in cyc_nx_bases[0][ci].nodes()], axis=0)
    G_cyc.add_node(ci, pos=pos, type=1, color=colors[0])
# add nodes for cycles in G2, type 2, position as the average of nodes
for ci in range(N_cyc2):
    pos = np.mean([graph_sets[1].nodes[i]['pos'] for i in cyc_nx_bases[1][ci].nodes()], axis=0)
    G_cyc.add_node(ci+N_cyc1, pos=pos, type=2, color=colors[1])

In [ ]:
# Compute linking number
# compute linkage of basis vectors
numeric_res = calc_basis_linkage(*cyc_nx_bases) # these numbers are close to integers (numerical errors)
lk_mat = extract_linkage_matrix(numeric_res)

In [ ]:
# Construct the network of cycles
for ci in range(N_cyc1):
    for cj in range(N_cyc2):
        if abs(lk_mat[ci, cj]) > 0:
            G_cyc.add_edge(ci, N_cyc1+cj, weight=lk_mat[ci, cj])

In [ ]:
# Visualize the network of cycles
plot_networkx_custom(G_cyc)

# Check for cycles - TODO: add labels of nodes + find the corresponding cycles
try:
    cycles = nx.cycle_basis(G_cyc)
    print("Cycle found!")

    for c in cycles:
        sign = 1
        for i in range(len(c)-1):
            sign *= G_cyc.edges[(c[i], c[i+1])]['weight']
        sign *= G_cyc.edges[(c[-1], c[0])]['weight']
        print(c, sign)
except nx.exception.NetworkXNoCycle:
    print("No cycles found.")

Cycle found!
[50, 8, 79, 15, 61, 11, 62, 13] 1.0
[22, 78, 27, 34, 15, 61, 11, 62, 13, 33, 25, 70] 1.0
[21, 107, 26, 78, 27, 34, 15, 61, 11, 62, 13, 33, 25, 70, 24, 108] 1.0
[28, 9, 44, 23, 29, 27, 34, 15] 1.0
[45, 9, 44, 23, 29, 27, 34, 15, 61, 11, 62, 13] 1.0
[50, 5, 42, 1, 62, 13] -1.0
[19, 37, 5, 42, 1, 62, 13, 33, 25, 70] -1.0
[19, 81, 16, 98, 2, 57, 0, 42, 1, 62, 13, 33, 25, 70] 1.0
[45, 6, 51, 20, 93, 18, 81, 16, 98, 2, 57, 0, 42, 1, 62, 13] 1.0
[17, 72, 16, 98, 2, 57, 0, 42, 1, 62, 13, 33, 25, 70, 24, 108] 1.0
[50, 12, 49, 0, 42, 1, 62, 13] 1.0
[53, 12, 49, 0, 42, 1, 62, 13, 33, 25, 70, 24] 1.0
[32, 3, 106, 12, 49, 0] -1.0
[17, 91, 3, 106, 12, 49, 0, 42, 1, 62, 13, 33, 25, 70, 24, 108] 1.0
